In [ ]:
import os,sys
import numpy as np
import gc
import scipy.io as scio
import tensorflow as tf
import keras
import random
from tensorflow.keras.layers import Input, Dense, Dropout, Layer, Conv2D, MaxPooling2D, Bidirectional, LSTM, GRU, Lambda
from tensorflow.keras.layers import TimeDistributed, Flatten, LeakyReLU, Softmax, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, LearningRateScheduler, ReduceLROnPlateau
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.regularizers import l2
from sklearn.metrics import accuracy_score
import keras_tuner as kt
import keras.backend as K
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split, KFold
import psutil
import time
from collections import defaultdict
import json
import os
from datetime import datetime

In [ ]:
# Parameters
use_existing_model = False
fraction_for_test = 0.20
data_bvp = 'BVPwithout2/' # Without dataset 2
ALL_MOTION = [1,2,3,4,5,6,7,8,9,10,11,12]
N_MOTION = len(ALL_MOTION)
T_MAX = 0
n_epochs = 40
f_dropout_ratio = 0.1
n_batch_size = 24
f_learning_rate = 0.001
n_gru_hidden_units = 128

In [ ]:
# Standardize activity labels
activity_mapping = {
    '20181109-VS': {1: 'Push&Pull', 2: 'Sweep', 3: 'Clap', 4: 'Slide', 5: 'Draw-Zigzag(Vertical)', 6: 'Draw-N(Vertical)'},
    '20181115-VS': {1: 'Push&Pull', 2: 'Sweep', 3: 'Clap', 4: 'Draw-O(Vertical)', 5: 'Draw-Zigzag(Vertical)', 6: 'Draw-N(Vertical)'},
    '20181117-VS': {1: 'Push&Pull', 2: 'Sweep', 3: 'Clap', 4: 'Draw-O(Vertical)', 5: 'Draw-Zigzag(Vertical)', 6: 'Draw-N(Vertical)'},
    '20181118-VS': {1: 'Push&Pull', 2: 'Sweep', 3: 'Clap', 4: 'Draw-O(Vertical)', 5: 'Draw-Zigzag(Vertical)', 6: 'Draw-N(Vertical)'},
    '20181121-VS': {1: 'Slide', 2: 'Draw-O(Horizontal)', 3: 'Draw-Zigzag(Horizontal)', 4: 'Draw-N(Horizontal)', 5: 'Draw-Triangle(Horizontal)', 6: 'Draw-Rectangle(Horizontal)'},
    '20181127-VS': {1: 'Slide', 2: 'Draw-O(Horizontal)', 3: 'Draw-Zigzag(Horizontal)', 4: 'Draw-N(Horizontal)', 5: 'Draw-Triangle(Horizontal)', 6: 'Draw-Rectangle(Horizontal)'},
    '20181128-VS': {1: 'Push&Pull', 2: 'Sweep', 3: 'Clap', 4: 'Draw-O(Horizontal)', 5: 'Draw-Zigzag(Horizontal)', 6: 'Draw-N(Horizontal)'},
    '20181130-VS': {1: 'Push&Pull', 2: 'Sweep', 3: 'Clap', 4: 'Slide', 5: 'Draw-O(Horizontal)', 6: 'Draw-Zigzag(Horizontal)', 7: 'Draw-N(Horizontal)', 8: 'Draw-Triangle(Horizontal)', 9: 'Draw-Rectangle(Horizontal)'},
    '20181204-VS': {1: 'Push&Pull', 2: 'Sweep', 3: 'Clap', 4: 'Slide', 5: 'Draw-O(Horizontal)', 6: 'Draw-Zigzag(Horizontal)', 7: 'Draw-N(Horizontal)', 8: 'Draw-Triangle(Horizontal)', 9: 'Draw-Rectangle(Horizontal)'},
    
    # ── SPECIAL SESSION: User-specific nested mapping ──────────────────────
    '20181205-VS': {
        'user2': {1: 'Draw-O(Horizontal)', 2: 'Draw-Zigzag(Horizontal)', 3: 'Draw-N(Horizontal)', 4: 'Draw-Triangle(Horizontal)', 5: 'Draw-Rectangle(Horizontal)'},
        'user3': {1: 'Slide', 2: 'Draw-O(Horizontal)', 3: 'Draw-Zigzag(Horizontal)', 4: 'Draw-N(Horizontal)', 5: 'Draw-Triangle(Horizontal)', 6: 'Draw-Rectangle(Horizontal)'}},
    
    '20181208-VS': {1: 'Push&Pull', 2: 'Sweep', 3: 'Clap', 4: 'Slide'},
    '20181209-VS': {1: 'Push&Pull', 2: 'Sweep', 3: 'Clap', 4: 'Slide', 5: 'Draw-O(Horizontal)', 6: 'Draw-Zigzag(Horizontal)'},
    '20181211-VS': {1: 'Push&Pull', 2: 'Sweep', 3: 'Clap', 4: 'Slide', 5: 'Draw-O(Horizontal)', 6: 'Draw-Zigzag(Horizontal)'}
}

# Create a consistent set of labels
standardized_labels = {
    'Push&Pull': 1,
    'Sweep': 2,
    'Clap': 3,
    'Slide': 4,
    'Draw-O(Horizontal)': 5,
    'Draw-Zigzag(Horizontal)': 6,
    'Draw-Zigzag(Vertical)': 7,
    'Draw-N(Vertical)': 8,
    'Draw-O(Vertical)': 9,
    'Draw-N(Horizontal)': 10,
    'Draw-Triangle(Horizontal)': 11,
    'Draw-Rectangle(Horizontal)': 12
}

# Session dates for chronological sorting
session_dates = {
    '20181109-VS': '20181109',
    '20181115-VS': '20181115',
    '20181117-VS': '20181117',
    '20181118-VS': '20181118',
    '20181121-VS': '20181121',
    '20181127-VS': '20181127',
    '20181128-VS': '20181128',
    '20181130-VS': '20181130',
    '20181204-VS': '20181204',
    '20181205-VS': '20181205',
    '20181208-VS': '20181208',
    '20181209-VS': '20181209',
    '20181211-VS': '20181211'
}

# Function to standardize activity labels
def standardize_activity_labels(date, activity_label, user_id):
    """
    Standardize activity labels with support for nested user-specific mappings.
    
    Args:
        date: Session date (e.g., '20181205-VS')
        activity_label: Original activity label (1-9)
        user_id: User ID (e.g., 'user2')
    
    Returns:
        Standardized label (1-12) or None if not found
    """
    if date not in activity_mapping:
        return None
    
    session_mapping = activity_mapping[date]
    
    # Check if this session has user-specific mappings
    first_key = next(iter(session_mapping.keys()))
    
    if isinstance(first_key, str):  # User-specific mapping
        # This is a special session like 20181205-VS
        if user_id not in session_mapping:
            return None
        
        user_mapping = session_mapping[user_id]
        if activity_label not in user_mapping:
            return None
        
        original_label = user_mapping[activity_label]
    else:  # Standard mapping
        if activity_label not in session_mapping:
            return None
        original_label = session_mapping[activity_label]
    
    if original_label in standardized_labels:
        return standardized_labels[original_label]
    
    return None


def determine_T_MAX(directories, motion_sel):
    T_MAX = 0
    for directory in directories:
        for data_root, data_dirs, data_files in os.walk(directory):
            for data_file_name in data_files:
                file_path = os.path.join(data_root, data_file_name)
                try:
                    data_1 = scio.loadmat(file_path)['velocity_spectrum_ro']
                    label_1 = int(data_file_name.split('-')[1])
                    if label_1 not in motion_sel:
                        continue
                    T_MAX = max(T_MAX, data_1.shape[2])
                except Exception:
                    continue
    return T_MAX

def normalize_data(data_1):
    data_1_max = np.concatenate((data_1.max(axis=0), data_1.max(axis=1)), axis=0).max(axis=0)
    data_1_min = np.concatenate((data_1.min(axis=0), data_1.min(axis=1)), axis=0).min(axis=0)
    if (len(np.where((data_1_max - data_1_min) == 0)[0]) > 0):
        return data_1
    data_1_max_rep = np.tile(data_1_max, (data_1.shape[0], data_1.shape[1], 1))
    data_1_min_rep = np.tile(data_1_min, (data_1.shape[0], data_1.shape[1], 1))
    data_1_norm = (data_1 - data_1_min_rep) / (data_1_max_rep - data_1_min_rep)
    return data_1_norm

def zero_padding(data, T_MAX):
    data_pad = []
    for i in range(len(data)):
        t = np.array(data[i]).shape[2]
        data_pad.append(np.pad(data[i], ((0, 0), (0, 0), (T_MAX - t, 0)), 'constant', constant_values=0).tolist())
    return np.array(data_pad)

def onehot_encoding(label, num_class):
    label = np.array(label).astype('int32')
    label = np.squeeze(label)
    _label = np.eye(num_class)[label - 1]  # from label to onehot
    return _label

def load_4D_data(directories, motion_sel, T_MAX, second_subfolder, user_id):
    """
    Load 4D data with user-specific activity label handling.
    """
    data = []
    label = []
    skipped_count = 0
    
    for directory in directories:
        for data_root, data_dirs, data_files in os.walk(directory):
            for data_file_name in data_files:
                file_path = os.path.join(data_root, data_file_name)
                try:
                    data_1 = scio.loadmat(file_path)['velocity_spectrum_ro']
                    label_1 = int(data_file_name.split('-')[1])
                    
                    # Standardize label (now with user_id)
                    standardized_label_1 = standardize_activity_labels(
                        second_subfolder, 
                        label_1, 
                        user_id
                    )
                    
                    if standardized_label_1 is None:
                        skipped_count += 1
                        continue
                    
                    if standardized_label_1 not in motion_sel:
                        continue

                    data_normed_1 = normalize_data(data_1)
                    
                    # Save to list
                    data.append(data_normed_1.tolist())
                    label.append(standardized_label_1)
                    
                except Exception as e:
                    continue
    
    if skipped_count > 0:
        print(f"  Skipped {skipped_count} samples (unmapped labels)")
    
    if len(data) == 0:
        # Return empty arrays with correct shape
        return np.empty((0, 38, 20, 20, 1)), np.empty((0, 12))
    
    data = zero_padding(data, T_MAX)
    data = np.swapaxes(np.swapaxes(data, 1, 3), 2, 3)
    data = np.expand_dims(data, axis=-1)

    # Convert labels to ndarray
    label = np.array(label)
    label = onehot_encoding(label, len(standardized_labels))    
    return data, label

In [ ]:
# Define room-user mapping
room_user_mapping = {
    'room1': ['user1', 'user2', 'user3', 'user5', 'user10', 'user11', 'user12', 'user13', 'user14', 'user15', 'user16', 'user17'],
    'room2': ['user1', 'user2', 'user3', 'user4', 'user5', 'user6'],
    'room3': ['user3', 'user7', 'user8', 'user9']
}

# Define subfolder-room mapping
subfolder_room_mapping = {
    '20181109-VS': 'room1',
    '20181115-VS': 'room1',
    '20181117-VS': 'room2',
    '20181118-VS': 'room2',
    '20181121-VS': 'room1',
    '20181127-VS': 'room2',
    '20181128-VS': 'room2',
    '20181130-VS': 'room1',
    '20181204-VS': 'room2',
    '20181205-VS': 'room2',
    '20181208-VS': 'room2',
    '20181209-VS': 'room2',
    '20181211-VS': 'room3'
}


def generate_domain_label(subfolder, room, user_id, criteria):
    if criteria == 'subfolder':
        return subfolder
    elif criteria == 'room':
        return room
    elif criteria == 'user':
        return user_id
    else:
        raise ValueError("Invalid criteria. Choose from 'subfolder', 'room', or 'user'.")

def get_user_rooms(user_id, room_user_mapping):
    rooms = []
    for room, users in room_user_mapping.items():
        if user_id in users:
            rooms.append(room)
    return rooms

def load_all_user_data(base_directory, room_user_mapping, subfolder_room_mapping):
    fl_clients = {}
    data_index = []

    print("Step 1: Determining T_MAX...")
    all_directories = []
    for root, dirs, files in os.walk(base_directory):
        for subfolder in dirs:
            subfolder_path = os.path.join(root, subfolder)
            if os.path.isdir(subfolder_path):
                for second_subfolder in os.listdir(subfolder_path):
                    second_subfolder_path = os.path.join(subfolder_path, second_subfolder)
                    if os.path.isdir(second_subfolder_path):
                        for user_dir in os.listdir(second_subfolder_path):
                            if 'user' in user_dir:
                                user_path = os.path.join(second_subfolder_path, user_dir)
                                all_directories.append(user_path)
    
    T_MAX = determine_T_MAX(all_directories, ALL_MOTION)
    print(f"Global T_MAX = {T_MAX}\n")
    

    print("Step 2: Loading data with session tracking...\n")
    # Temporary storage: {user_id: {room: [session_data_list]}}
    temp_storage = {}
    
    for root, dirs, files in os.walk(base_directory):
        for subfolder in dirs:
            subfolder_path = os.path.join(root, subfolder)
            if os.path.isdir(subfolder_path):
                for second_subfolder in sorted(os.listdir(subfolder_path)):
                    second_subfolder_path = os.path.join(subfolder_path, second_subfolder)
                    if os.path.isdir(second_subfolder_path):
                        
                        # Check if this is a valid session
                        if second_subfolder not in subfolder_room_mapping:
                            continue
                        
                        room = subfolder_room_mapping[second_subfolder]
                        session_date = session_dates.get(second_subfolder, second_subfolder[:8])
                        
                        for user_dir in sorted(os.listdir(second_subfolder_path)):
                            if 'user' in user_dir:
                                user_id = user_dir
                                user_path = os.path.join(second_subfolder_path, user_dir)
                                
                                # Check if user should be in this room
                                rooms = get_user_rooms(user_id, room_user_mapping)
                                if room not in rooms:
                                    continue
                                
                                # Load data for this session
                                print(f"  Loading {user_id} from {second_subfolder} ({room})...", end=" ")
                                
                                data, label = load_4D_data(
                                    [user_path], 
                                    ALL_MOTION, 
                                    T_MAX, 
                                    second_subfolder, 
                                    user_id
                                )
                                
                                if len(data) == 0:
                                    print(f"  No valid data")
                                    continue
                                
                                print(f"✓ {len(data)} samples")
                                
                                # Initialize storage
                                if user_id not in temp_storage:
                                    temp_storage[user_id] = {}
                                if room not in temp_storage[user_id]:
                                    temp_storage[user_id][room] = []
                                
                                # Store session data
                                temp_storage[user_id][room].append({
                                    'session_name': second_subfolder,
                                    'session_date': session_date,
                                    'data': data,
                                    'labels': label,
                                    'n_samples': len(data)
                                })
    
    print("Step 3: Sorting sessions chronologically and concatenating...\n")
    
    for user_id in sorted(temp_storage.keys()):
        fl_clients[user_id] = {}
        
        print(f"── {user_id} ──")
        
        for room in sorted(temp_storage[user_id].keys()):
            # Sort sessions by date
            sessions = temp_storage[user_id][room]
            sessions.sort(key=lambda x: x['session_date'])
            
            print(f"  {room} - Chronological order:")
            for i, session in enumerate(sessions, 1):
                print(f"    {i}. {session['session_date']} - {session['session_name']}: "
                      f"{session['n_samples']} samples")
            
            # Concatenate data chronologically
            all_data = []
            all_labels = []
            session_tags = []
            session_metadata = []
            current_idx = 0
            
            for session in sessions:
                all_data.append(session['data'])
                all_labels.append(session['labels'])
                
                # Tag each sample with its session
                n_samples = session['n_samples']
                session_tags.extend([session['session_name']] * n_samples)
                
                # Store session metadata with indices
                session_metadata.append({
                    'session_name': session['session_name'],
                    'session_date': session['session_date'],
                    'n_samples': n_samples,
                    'start_idx': current_idx,
                    'end_idx': current_idx + n_samples
                })
                current_idx += n_samples
            
            # Store in fl_clients
            fl_clients[user_id][room] = {
                'data': np.concatenate(all_data, axis=0),
                'labels': np.concatenate(all_labels, axis=0),
                'session_tags': np.array(session_tags),
                'sessions': session_metadata
            }
            
            total_samples = len(fl_clients[user_id][room]['data'])
            print(f"  → Total: {total_samples} samples (chronologically ordered)\n")
    

    total_samples = 0
    total_sessions = 0
    
    for user_id in sorted(fl_clients.keys()):
        print(f"Client: {user_id}")
        for room in sorted(fl_clients[user_id].keys()):
            data_shape = fl_clients[user_id][room]['data'].shape
            labels_shape = fl_clients[user_id][room]['labels'].shape
            n_sessions = len(fl_clients[user_id][room]['sessions'])
            
            total_samples += data_shape[0]
            total_sessions += n_sessions
            
            print(f"  {room}: {data_shape[0]} samples from {n_sessions} session(s)")
            print(f"    Data: {data_shape}, Labels: {labels_shape}")
    
    print(f"\n{'─'*80}")
    print(f"TOTAL: {total_samples} samples from {total_sessions} sessions across "
          f"{len(fl_clients)} users")
    print(f"{'='*80}\n")
    return fl_clients, data_index

fl_clients, data_index = load_all_user_data(data_bvp, room_user_mapping, subfolder_room_mapping)

In [ ]:
# Prepare train/val/test split
def prepare_grouped_baseline_data(fl_clients, initial_clients, new_clients,
                                   train_ratio=0.70, val_ratio=0.15,
                                   test_ratio=0.15, seed=42):

    def split_single_group(group_dict, group_name):
        np.random.seed(seed)
        per_user_splits = {}

        all_train_data,  all_train_labels = [], []
        all_val_data,    all_val_labels   = [], []
        all_test_data,   all_test_labels  = [], []

        group_users = set()
        for room_users in group_dict.values():
            group_users.update(room_users)

        for user_id in sorted(group_users):
            if user_id not in fl_clients:
                continue

            user_rooms_in_group = [
                room for room, users in group_dict.items()
                if user_id in users and room in fl_clients[user_id]
            ]

            all_sessions = []
            for room in user_rooms_in_group:
                sessions = fl_clients[user_id][room]['sessions']
                data     = fl_clients[user_id][room]['data']
                labels   = fl_clients[user_id][room]['labels']

                for session in sessions:
                    start_idx = session['start_idx']
                    end_idx   = session['end_idx']
                    all_sessions.append({
                        'session_name': session['session_name'],
                        'session_date': session['session_date'],
                        'room':         room,
                        'data':         data[start_idx:end_idx],
                        'labels':       labels[start_idx:end_idx],
                        'n_samples':    session['n_samples']
                    })

            all_sessions.sort(key=lambda x: x['session_date'])
            n_sessions = len(all_sessions)

            for i, session in enumerate(all_sessions, 1):
                print(f"    {i}. {session['session_date']} - {session['session_name']} "
                      f"({session['room']}): {session['n_samples']} samples")

            # STEP 2: Split strategy
            if n_sessions >= 3:
                n_train_sessions = max(1, int(train_ratio * n_sessions))
                n_val_sessions   = max(1, int(val_ratio   * n_sessions))

                train_sessions = all_sessions[:n_train_sessions]
                val_sessions   = all_sessions[n_train_sessions:n_train_sessions + n_val_sessions]
                test_sessions  = all_sessions[n_train_sessions + n_val_sessions:]

                train_data   = np.concatenate([s['data']   for s in train_sessions], axis=0)
                train_labels = np.concatenate([s['labels'] for s in train_sessions], axis=0)

                val_data   = np.concatenate([s['data']   for s in val_sessions],  axis=0) if val_sessions  else np.empty((0, 38, 20, 20, 1))
                val_labels = np.concatenate([s['labels'] for s in val_sessions],  axis=0) if val_sessions  else np.empty((0, 12))

                test_data   = np.concatenate([s['data']   for s in test_sessions], axis=0) if test_sessions else np.empty((0, 38, 20, 20, 1))
                test_labels = np.concatenate([s['labels'] for s in test_sessions], axis=0) if test_sessions else np.empty((0, 12))

                session_info   = {
                    'train_sessions': [s['session_name'] for s in train_sessions],
                    'val_sessions':   [s['session_name'] for s in val_sessions],
                    'test_sessions':  [s['session_name'] for s in test_sessions]
                }
                split_strategy = 'session-based'

            elif n_sessions == 2:
                train_data   = all_sessions[0]['data'].copy()
                train_labels = all_sessions[0]['labels'].copy()

                second_data   = all_sessions[1]['data']
                second_labels = all_sessions[1]['labels']
                # Ensures val and test both see all classes from session 2
                class_indices  = np.argmax(second_labels, axis=1)
                unique_classes = np.unique(class_indices)

                val_parts,  val_lbl_parts  = [], []
                test_parts, test_lbl_parts = [], []

                for cls in unique_classes:
                    mask     = class_indices == cls
                    cls_d    = second_data[mask]
                    cls_l    = second_labels[mask]
                    n_half   = max(1, len(cls_d) // 2)
                    val_parts.append(cls_d[:n_half])
                    val_lbl_parts.append(cls_l[:n_half])
                    test_parts.append(cls_d[n_half:])
                    test_lbl_parts.append(cls_l[n_half:])


                val_data   = np.concatenate(val_parts,  axis=0)
                val_labels = np.concatenate(val_lbl_parts,  axis=0)
                test_data  = np.concatenate(test_parts, axis=0)
                test_labels= np.concatenate(test_lbl_parts, axis=0)

                del val_parts, val_lbl_parts, test_parts, test_lbl_parts
                del class_indices

                session_info   = {
                    'train_sessions': [all_sessions[0]['session_name']],
                    'val_sessions':   [all_sessions[1]['session_name']],
                    'test_sessions':  [all_sessions[1]['session_name']]
                }
                split_strategy = 'hybrid-session-sample'

            else:  # n_sessions == 1
                all_user_data   = np.concatenate([s['data']   for s in all_sessions], axis=0)
                all_user_labels = np.concatenate([s['labels'] for s in all_sessions], axis=0)

                train_parts, train_label_parts = [], []
                val_parts,   val_label_parts   = [], []
                test_parts,  test_label_parts  = [], []

                class_indices  = np.argmax(all_user_labels, axis=1)
                unique_classes = sorted(np.unique(class_indices))
                print(f"  Activities found: {unique_classes} ({len(unique_classes)} total)")

                for class_idx in unique_classes:
                    mask        = class_indices == class_idx
                    class_data  = all_user_data[mask]
                    class_lbls  = all_user_labels[mask]
                    n_c         = len(class_data)
                    n_train     = int(train_ratio * n_c)
                    n_val_c     = int(val_ratio   * n_c)

                    train_parts.append(class_data[:n_train])
                    train_label_parts.append(class_lbls[:n_train])
                    val_parts.append(class_data[n_train:n_train + n_val_c])
                    val_label_parts.append(class_lbls[n_train:n_train + n_val_c])
                    test_parts.append(class_data[n_train + n_val_c:])
                    test_label_parts.append(class_lbls[n_train + n_val_c:])

                train_data   = np.concatenate(train_parts, axis=0)
                train_labels = np.concatenate(train_label_parts, axis=0)
                val_data     = np.concatenate(val_parts,   axis=0)
                val_labels   = np.concatenate(val_label_parts,   axis=0)
                test_data    = np.concatenate(test_parts,  axis=0)
                test_labels  = np.concatenate(test_label_parts,  axis=0)

                # Free intermediate class-split lists immediately
                del train_parts, train_label_parts
                del val_parts,   val_label_parts
                del test_parts,  test_label_parts
                del all_user_data, all_user_labels, class_indices

                session_info   = {
                    'train_sessions': [s['session_name'] for s in all_sessions],
                    'val_sessions':   [s['session_name'] for s in all_sessions],
                    'test_sessions':  [s['session_name'] for s in all_sessions]
                }
                split_strategy = 'stratified'

            # Free all_sessions immediately after splits are built
            del all_sessions
            gc.collect()

            # STEP 3: Statistics
            n_total          = len(train_data) + len(val_data) + len(test_data)
            actual_train_pct = len(train_data) / n_total * 100 if n_total > 0 else 0
            actual_val_pct   = len(val_data)   / n_total * 100 if n_total > 0 else 0
            actual_test_pct  = len(test_data)  / n_total * 100 if n_total > 0 else 0

            # STEP 4: Class distribution check
            train_class_dist = (np.bincount(np.argmax(train_labels, axis=1), minlength=12)
                                if len(train_data) > 0 else np.zeros(12, dtype=int))
            val_class_dist   = (np.bincount(np.argmax(val_labels,   axis=1), minlength=12)
                                if len(val_data)   > 0 else np.zeros(12, dtype=int))
            test_class_dist  = (np.bincount(np.argmax(test_labels,  axis=1), minlength=12)
                                if len(test_data)  > 0 else np.zeros(12, dtype=int))

            if len(train_data) > 0:
                print(f"    Train class dist: {train_class_dist}")
            if len(val_data)   > 0:
                print(f"    Val   class dist: {val_class_dist}")
            if len(test_data)  > 0:
                print(f"    Test  class dist: {test_class_dist}")

            if len(train_data) > 0 and len(val_data) > 0 and len(test_data) > 0:
                train_classes    = set(np.where(train_class_dist > 0)[0])
                val_classes      = set(np.where(val_class_dist   > 0)[0])
                test_classes     = set(np.where(test_class_dist  > 0)[0])
                missing_in_val   = train_classes - val_classes
                missing_in_test  = train_classes - test_classes

                if missing_in_val:
                    print(f"    ⚠️  Classes in train but NOT in val : "
                          f"{sorted(missing_in_val)}")
                if missing_in_test:
                    print(f"    ⚠️  Classes in train but NOT in test: "
                          f"{sorted(missing_in_test)}")

            # STEP 5: Store splits
            per_user_splits[user_id] = {
                'train_data':    train_data,
                'train_labels':  train_labels,
                'val_data':      val_data,
                'val_labels':    val_labels,
                'test_data':     test_data,
                'test_labels':   test_labels,
                'total_samples': n_total,
                'train_samples': len(train_data),
                'val_samples':   len(val_data),
                'test_samples':  len(test_data),
                'n_sessions':    n_sessions,
                'split_strategy':split_strategy,
                'rooms':         user_rooms_in_group,
                **session_info
            }

            # Append references for group aggregation
            all_train_data.append(train_data)
            all_train_labels.append(train_labels)
            all_val_data.append(val_data)
            all_val_labels.append(val_labels)
            all_test_data.append(test_data)
            all_test_labels.append(test_labels)

        # GROUP AGGREGATION
        group_splits = {
            'train_data':   np.concatenate(all_train_data,   axis=0),
            'train_labels': np.concatenate(all_train_labels, axis=0),
            'val_data':     np.concatenate(all_val_data,     axis=0),
            'val_labels':   np.concatenate(all_val_labels,   axis=0),
            'test_data':    np.concatenate(all_test_data,    axis=0),
            'test_labels':  np.concatenate(all_test_labels,  axis=0)
        }

        del all_train_data,  all_train_labels
        del all_val_data,    all_val_labels
        del all_test_data,   all_test_labels
        gc.collect()

        # Group class distribution
        group_train_dist = np.bincount(
            np.argmax(group_splits['train_labels'], axis=1), minlength=12)
        group_val_dist   = np.bincount(
            np.argmax(group_splits['val_labels'],   axis=1), minlength=12)
        group_test_dist  = np.bincount(
            np.argmax(group_splits['test_labels'],  axis=1), minlength=12)

        total_group         = (len(group_splits['train_data']) +
                               len(group_splits['val_data'])   +
                               len(group_splits['test_data']))
        actual_group_train  = len(group_splits['train_data']) / total_group * 100
        actual_group_val    = len(group_splits['val_data'])   / total_group * 100
        actual_group_test   = len(group_splits['test_data'])  / total_group * 100

        session_based_users = [uid for uid, s in per_user_splits.items()
                               if s['split_strategy'] == 'session-based']
        hybrid_users        = [uid for uid, s in per_user_splits.items()
                               if s['split_strategy'] == 'hybrid-session-sample']
        stratified_users    = [uid for uid, s in per_user_splits.items()
                               if s['split_strategy'] == 'stratified']

        group_train_classes = set(np.where(group_train_dist > 0)[0])
        group_val_classes   = set(np.where(group_val_dist   > 0)[0])
        group_test_classes  = set(np.where(group_test_dist  > 0)[0])

        if group_train_classes == group_val_classes == group_test_classes:
            print(f"\n  All splits have complete class coverage in this group")
        else:
            missing_in_val  = group_train_classes - group_val_classes
            missing_in_test = group_train_classes - group_test_classes
            if missing_in_val:
                print(f"\n  ⚠️  Group classes in train but NOT in val: {missing_in_val}")
            if missing_in_test:
                print(f"  ⚠️  Group classes in train but NOT in test: {missing_in_test}")

        return per_user_splits, group_splits

    group1_per_user, group1_splits = split_single_group(
        initial_clients, "GROUP 1 — Initial Clients")
    group2_per_user, group2_splits = split_single_group(
        new_clients, "GROUP 2 — New Clients")

    total_train = len(group1_splits['train_data']) + len(group2_splits['train_data'])
    total_val   = len(group1_splits['val_data'])   + len(group2_splits['val_data'])
    total_test  = len(group1_splits['test_data'])  + len(group2_splits['test_data'])
    total_all   = total_train + total_val + total_test

    return group1_per_user, group1_splits, group2_per_user, group2_splits


def get_data_split_for_seed(seed, fl_clients, 
                            train_ratio=0.70, val_ratio=0.15, test_ratio=0.15,
                            case='case1'):

    if case == 'case1':
        # Case 1: Imbalanced users (G2) join balanced users (G1)
        initial_clients = {
            'room1': ['user10', 'user11', 'user12', 'user13', 'user14', 'user15', 'user16', 'user17'],
            'room2': ['user4', 'user6'],
            'room3': ['user7', 'user8', 'user9']
        }
        new_clients = {
            'room1': ['user1', 'user2', 'user3', 'user5'],
            'room2': ['user1', 'user2', 'user3', 'user5'],
            'room3': ['user3']
        }
        
    elif case == 'case2':
        # Case 2: Balanced users (G2) join imbalanced users (G1)
        # (Swapped from case1)
        initial_clients = {
            'room1': ['user1', 'user2', 'user3', 'user5'],
            'room2': ['user1', 'user2', 'user3', 'user5'],
            'room3': ['user3']
        }
        new_clients = {
            'room1': ['user10', 'user11', 'user12', 'user13', 'user14', 'user15', 'user16', 'user17'],
            'room2': ['user4', 'user6'],
            'room3': ['user7', 'user8', 'user9']
        }


        
    elif case == 'case3':
        # Case 3: Cross-Room Generalization
        initial_clients = {
                'room1': ['user1', 'user2', 'user5', 'user10', 'user11', 'user12', 'user13', 'user14', 'user15', 'user16', 'user17'],
                'room2': ['user1', 'user2', 'user4', 'user5', 'user6']
                }
        new_clients = {
                'room3': ['user7', 'user8', 'user9']
                }
        
    else:
        raise ValueError(f"Unknown case: {case}. Must be 'case1', 'case2', or 'case3'")
    

    # VALIDATE: No user overlap between groups
    g1_users = set()
    for room_users in initial_clients.values():
        g1_users.update(room_users)
    
    g2_users = set()
    for room_users in new_clients.values():
        g2_users.update(room_users)
    
    overlap = g1_users & g2_users
    if overlap:
        raise ValueError(f"❌ Invalid configuration for {case}: "
                        f"Users {overlap} appear in both groups!")
    
    print(f"\n{'='*80}")
    print(f"  CASE: {case.upper()}")
    print(f"{'='*80}")
    print(f"  G1 (Initial) users: {sorted(g1_users)}")
    print(f"  G2 (New) users:     {sorted(g2_users)}")
    print(f"  No overlap: ")
    print(f"{'='*80}\n")
    

    # PREPARE DATA SPLITS
    group1_per_user, group1_splits, \
    group2_per_user, group2_splits = prepare_grouped_baseline_data(
        fl_clients=fl_clients,
        initial_clients=initial_clients,
        new_clients=new_clients,
        train_ratio=train_ratio,
        val_ratio=val_ratio,
        test_ratio=test_ratio,
        seed=seed)
    
    return {
        'group1_per_user': group1_per_user,
        'group1_splits':   group1_splits,
        'group2_per_user': group2_per_user,
        'group2_splits':   group2_splits,
        'case':            case,
        'g1_users':        sorted(g1_users),
        'g2_users':        sorted(g2_users)
    }


In [ ]:
# DataGenerators
class LazySeedDataLoader:
    def __init__(self, fl_clients, case='case1',
                 train_ratio=0.70, val_ratio=0.15, test_ratio=0.15):
        self.fl_clients  = fl_clients
        self.case        = case
        self.train_ratio = train_ratio
        self.val_ratio   = val_ratio
        self.test_ratio  = test_ratio
        self._current_seed = None
        self._current_data = None
        valid_cases = ['case1', 'case2', 'case3']
        if case not in valid_cases:
            raise ValueError(f"Invalid case: {case}. Must be one of {valid_cases}")
        

    def load(self, seed):
        if self._current_seed == seed and self._current_data is not None:
            print(f"  Seed {seed} already loaded — reusing")
            return self._current_data

        if self._current_data is not None:
            self._free_current()

        print(f"  Loading data for seed {seed} ({self.case})...")
        self._current_data = get_data_split_for_seed(
            seed=seed,
            fl_clients=self.fl_clients,
            train_ratio=self.train_ratio,
            val_ratio=self.val_ratio,
            test_ratio=self.test_ratio,
            case=self.case)  
        
        self._current_seed = seed
        g1 = self._current_data['group1_per_user']
        g2 = self._current_data['group2_per_user']
        
        g1_train = sum(len(s['train_data']) for s in g1.values())
        g2_train = sum(len(s['train_data']) for s in g2.values())
        
        print(f"  Seed {seed} loaded:")
        print(f"     G1 users : {sorted(g1.keys())}")
        print(f"     G2 users : {sorted(g2.keys())}")
        print(f"     G1 train : {g1_train} samples")
        print(f"     G2 train : {g2_train} samples")

        return self._current_data

    def _free_current(self):
        if self._current_data is None:
            return
        freed_seed = self._current_seed
        # Free per-user splits
        for group_key in ['group1_per_user', 'group2_per_user']:
            per_user = self._current_data.get(group_key, {})
            for uid, splits in per_user.items():
                for arr_key in ['train_data', 'train_labels',
                               'val_data',   'val_labels',
                               'test_data',  'test_labels']:
                    if arr_key in splits:
                        del splits[arr_key]

        # Free group aggregated splits
        for group_key in ['group1_splits', 'group2_splits']:
            splits = self._current_data.get(group_key, {})
            for arr_key in ['train_data', 'train_labels',
                           'val_data',   'val_labels',
                           'test_data',  'test_labels']:
                if arr_key in splits:
                    del splits[arr_key]

        del self._current_data
        self._current_data = None
        self._current_seed = None

        gc.collect()
        print(f"  Freed data for seed {freed_seed}")

    def __del__(self):
        self._free_current()



class FedPerDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, data, labels, batch_size, shuffle=True):
        self.data       = data    
        self.labels     = labels  
        self.batch_size = batch_size
        self.shuffle    = shuffle
        self.indexes    = np.arange(len(self.data))
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __len__(self):
        return int(np.ceil(len(self.data) / self.batch_size))

    def __getitem__(self, index):
        start         = index * self.batch_size
        end           = min(start + self.batch_size, len(self.data))
        batch_indexes = self.indexes[start:end]
        return self.data[batch_indexes], self.labels[batch_indexes]

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)


class FedPerPredictGenerator(tf.keras.utils.Sequence):
    def __init__(self, data, batch_size=32):
        self.data       = data
        self.batch_size = batch_size
        self.indexes    = np.arange(len(data))

    def __len__(self):
        return int(np.ceil(len(self.data) / self.batch_size))

    def __getitem__(self, index):
        start = index * self.batch_size
        end   = min(start + self.batch_size, len(self.data))
        return self.data[self.indexes[start:end]]


# Evaluation using PredictGenerator
def evaluate_with_generator(model, test_data, test_labels,
                             batch_size=32,
                             output_key='global_output'):
    """
    GPU-safe evaluation.
    Uses model(batch, training=False) instead of model.predict()
    to avoid cuDNN algorithm profiling that causes OOM after
    many training rounds.
    """
    n_samples  = len(test_data)
    n_batches  = max(1, (n_samples + batch_size - 1) // batch_size)

    all_preds  = []
    all_labels = []

    try:
        for batch_idx in range(n_batches):
            start   = batch_idx * batch_size
            end     = min(start + batch_size, n_samples)

            # ── numpy slice, only this batch goes to GPU ───────────────
            batch_x = test_data[start:end]
            batch_y = test_labels[start:end]

            # ── direct call: NO cuDNN profiling, NO Keras overhead ─────
            preds = model(batch_x, training=False)

            # ── handle multi-output ────────────────────────────────────
            if isinstance(preds, dict):
                if output_key and output_key in preds:
                    preds = preds[output_key]
                else:
                    key   = list(preds.keys())[0]
                    preds = preds[key]
            elif isinstance(preds, (list, tuple)):
                out_names = model.output_names
                if output_key and output_key in out_names:
                    preds = preds[out_names.index(output_key)]
                else:
                    preds = preds[0]

            all_preds.append(np.argmax(preds.numpy(), axis=1))
            all_labels.append(np.argmax(batch_y,      axis=1))

            del batch_x, batch_y, preds

        all_preds  = np.concatenate(all_preds)
        all_labels = np.concatenate(all_labels)
        accuracy   = float(np.mean(all_preds == all_labels))

        del all_preds, all_labels
        gc.collect()
        return accuracy

    except tf.errors.ResourceExhaustedError as e:
        print(f"    GPU OOM in evaluate_with_generator: {e}")
        del all_preds, all_labels
        gc.collect()
        return 0.0

    except Exception as e:
        print(f"    Eval error: {e}")
        gc.collect()
        return 0.0
    

# CUSTOM DATA GENERATOR FOR DUAL-HEAD MODELS
class FedPerDualHeadGenerator(tf.keras.utils.Sequence):
    def __init__(self, data, labels, batch_size, shuffle=True):
        self.data = data
        self.labels = labels
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.data))
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __len__(self):
        return int(np.ceil(len(self.data) / self.batch_size))

    def __getitem__(self, index):
        start = index * self.batch_size
        end = min(start + self.batch_size, len(self.data))
        batch_indexes = self.indexes[start:end]
        batch_x = self.data[batch_indexes]
        batch_y = self.labels[batch_indexes]
        return batch_x, {'global_output': batch_y, 'personal_output': batch_y}

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

In [ ]:
# Add Memory Management Utilities  
# Safely delete a model and free its memory
def safe_model_delete(model):
    if model is None:
        return
    try:
        # Clear sub-models first (dual-head case)
        if hasattr(model, 'base_model'):
            try:
                del model.base_model
            except Exception:
                pass

        # Release layer references
        if hasattr(model, 'layers'):
            for layer in model.layers:
                try:
                    # Clear layer weights from memory
                    if hasattr(layer, '_trainable_weights'):
                        layer._trainable_weights.clear()
                    if hasattr(layer, '_non_trainable_weights'):
                        layer._non_trainable_weights.clear()
                except Exception:
                    pass

        del model
    except Exception:
        pass
    gc.collect()


In [ ]:
# Loss Functions
# Example usage for Federated Learning
input_shape = (38, 20, 20, 1)
n_class = 12
# Weighted Loss for Class Imbalance
def get_class_weights(train_labels, epsilon=1e-8):
    class_counts = np.sum(train_labels, axis=0)
    n_samples = np.sum(class_counts)
    active_mask = class_counts > 0
    n_active = np.sum(active_mask)

    weights = np.zeros(len(class_counts))
    for c in range(len(class_counts)):
        if class_counts[c] > 0:
            weights[c] = n_samples / (n_active * class_counts[c])
        else:
            weights[c] = epsilon

    print(f" Class weights: {np.round(weights, 2)}")
    return weights

# Label smoothing for low severity users
def weighted_loss_with_smoothing(class_weights, smoothing=0.1):
    class_weights_tensor = tf.constant(class_weights, dtype=tf.float32)
    def loss_fn(y_true, y_pred):
        n_classes = tf.cast(tf.shape(y_true)[-1], tf.float32)
        y_true_smooth = y_true * (1.0 - smoothing) + (smoothing / n_classes)
        y_pred_clipped = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        per_class_loss = -y_true_smooth * tf.math.log(y_pred_clipped)  # (B, C)
        weighted_loss = per_class_loss * class_weights_tensor           # (B, C)
        return tf.reduce_mean(tf.reduce_sum(weighted_loss, axis=-1))
    return loss_fn


# HELPER FUNCTIONS FOR DUAL-HEAD
def sync_full_model_weights(local_model, global_model):
    synced = 0
    for layer in global_model.layers:
        try:
            local_layer = local_model.get_layer(layer.name)
            ws = layer.get_weights()
            local_ws = local_layer.get_weights()
            if ws and local_ws and [w.shape for w in ws] == [w.shape for w in local_ws]:
                local_layer.set_weights(ws)
                synced += 1
        except (ValueError, IndexError):
            pass
    return synced


def get_global_output_weights(model):
    try:
        layer = model.get_layer('global_output')
        ws = layer.get_weights()
        return [w.copy() for w in ws] if ws else None
    except ValueError:
        return None


def set_global_output_weights(model, weights):
    try:
        layer = model.get_layer('global_output')
        current = layer.get_weights()
        if current and [w.shape for w in current] == [w.shape for w in weights]:
            layer.set_weights(weights)
            return True
    except ValueError:
        pass
    return False

In [ ]:
# Severity Function (Stable vs Skewed clients)
def detect_severity_distribution_only(train_labels, val_labels, n_class=12):
    train_counts = np.sum(train_labels, axis=0)
    val_counts = np.sum(val_labels, axis=0)
    train_classes = set(np.where(train_counts > 0)[0])
    val_classes = set(np.where(val_counts > 0)[0])
    overlap_classes = train_classes & val_classes

    train_coverage = len(train_classes) / n_class
    val_coverage = len(val_classes) / n_class

    if len(train_classes) > 0 and len(val_classes) > 0:
        train_to_val_overlap = len(overlap_classes) / len(train_classes)
        val_to_train_overlap = len(overlap_classes) / len(val_classes)
        overlap_ratio = min(train_to_val_overlap, val_to_train_overlap)
    else:
        train_to_val_overlap = 1.0
        val_to_train_overlap = 1.0
        overlap_ratio = 1.0

    train_dist = train_counts / (train_counts.sum() + 1e-8)
    val_dist = val_counts / (val_counts.sum() + 1e-8)
    kl_divergence = np.sum(val_dist * np.log((val_dist + 1e-8) / (train_dist + 1e-8)))

    severity_score = 0.0
    if train_coverage < 0.5:     severity_score += 20
    elif train_coverage < 0.7:   severity_score += 10
    elif train_coverage < 0.9:   severity_score += 5
    if overlap_ratio < 0.3:      severity_score += 35
    elif overlap_ratio < 0.5:    severity_score += 25
    elif overlap_ratio < 0.7:    severity_score += 15
    elif overlap_ratio < 0.9:    severity_score += 5
    if kl_divergence > 2.0:      severity_score += 25
    elif kl_divergence > 1.0:    severity_score += 15
    elif kl_divergence > 0.5:    severity_score += 8
    if val_coverage < 0.3:       severity_score += 15
    elif val_coverage < 0.5:     severity_score += 8

    # (critical,high,moderate = Skewed)
    # (Low = Stable)
    if severity_score >= 70:     severity = 'CRITICAL'
    elif severity_score >= 50:   severity = 'HIGH'
    elif severity_score >= 30:   severity = 'MODERATE'
    else:                        severity = 'LOW'

    confidence = min(1.0, severity_score / 100.0) if severity != 'LOW' else 1.0 - (severity_score / 100.0)

    return severity, {
        'severity': severity, 'confidence': confidence, 'score': severity_score,
        'overlap_ratio': overlap_ratio, 'train_to_val_overlap': train_to_val_overlap,
        'val_to_train_overlap': val_to_train_overlap, 'train_coverage': train_coverage,
        'val_coverage': val_coverage, 'kl_divergence': kl_divergence,
        'n_train_classes': len(train_classes), 'n_val_classes': len(val_classes),
        'n_overlap_classes': len(overlap_classes),
    }



In [ ]:
BACKBONE_LAYER_NAMES = ['bb_conv', 'bb_td_dense', 'bb_bigru', 'bb_lstm']
PERSONAL_SEVERITIES  = ['LOW'] # Stable
GLOBAL_SEVERITIES    = ['MODERATE', 'HIGH', 'CRITICAL'] # Skewed
# and run_lightweight_trial (progressive exploration)
HEAD_CONFIGS = [
    {'d1': 64,  'd2': None},    # config 1: simplest, 1 layer small
    {'d1': 64,  'd2': 32},      # config 2: 2 layers narrow
    {'d1': 128, 'd2': None},    # config 3: 1 layer medium
    {'d1': 128, 'd2': 64},      # config 4: 2 layers medium
    {'d1': 256, 'd2': None},    # config 5: 1 layer wide
    {'d1': 256, 'd2': 128},     # config 6: 2 layers wide
]

# Shared backbone (Global model)
def shared_backbone(input_shape):
    model_input = Input(shape=input_shape, dtype='float32', name='bb_input')
    x = TimeDistributed(Conv2D(16, kernel_size=(5, 5), activation='relu',data_format='channels_last'),name='bb_conv')(model_input)
    x = TimeDistributed(MaxPooling2D(pool_size=(2, 2)),name='bb_pool')(x)
    x = TimeDistributed(Flatten(), name='bb_td_flatten')(x)
    x = TimeDistributed(Dense(128, activation='gelu',kernel_regularizer=tf.keras.regularizers.l2(0.01)),name='bb_td_dense')(x)
    x = Bidirectional(GRU(128, return_sequences=True,kernel_regularizer=tf.keras.regularizers.l2(0.05)),name='bb_bigru')(x)
    x = Dropout(0.2, name='bb_drop1')(x)
    x = LSTM(128, return_sequences=True,kernel_regularizer=tf.keras.regularizers.l2(0.01),name='bb_lstm')(x)
    x = Dropout(0.2, name='bb_drop2')(x)
    return model_input, x


def compile_global_model(input_shape, n_class, class_weights=None):
    shared_input, shared_output = shared_backbone(input_shape)
    x  = Flatten(name='bb_flatten')(shared_output)
    model_output = Dense(n_class, activation='softmax',name='global_output')(x)
    global_model = Model(inputs=shared_input, outputs=model_output)

    if class_weights is not None:
        loss_fn = weighted_loss_with_smoothing(class_weights, smoothing=0.1)
    else:
        loss_fn = CategoricalCrossentropy(label_smoothing=0.05)

    global_model.compile(
        optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.001),
        loss=loss_fn,
        metrics=['accuracy'])

    del loss_fn
    gc.collect()
    return global_model

# Backbone weight helpers 
def get_backbone_weights(model):
    backbone_weights = {}
    for layer_name in BACKBONE_LAYER_NAMES:
        try:
            layer   = model.get_layer(layer_name)
            weights = layer.get_weights()
            if weights:
                backbone_weights[layer_name] = [w.copy() for w in weights]
        except ValueError:
            continue
    return backbone_weights


def set_backbone_weights(model, backbone_weights):
    updated = 0
    for layer_name, weights in backbone_weights.items():
        try:
            layer   = model.get_layer(layer_name)
            current = layer.get_weights()
            if (current and
                    [w.shape for w in current] == [w.shape for w in weights]):
                layer.set_weights(weights)
                updated += 1
        except ValueError:
            continue
    return updated


# Arch string helper 
def _arch_str(cfg):
    if cfg['d2'] is not None:
        return f"({cfg['d1']},{cfg['d2']})"
    return f"({cfg['d1']},)"


# Dual-head compile 
def _compile_dual_head_model(model,
                              global_weight=0.3,
                              personal_weight=0.7):

    loss_fn = CategoricalCrossentropy(label_smoothing=0.1)
    model.compile(
        optimizer=tf.keras.optimizers.RMSprop(0.001),
        loss={'global_output':   loss_fn,
              'personal_output': loss_fn},
        loss_weights={'global_output':   global_weight,
                      'personal_output': personal_weight},
        metrics={'personal_output': 'accuracy'})
    model._custom_loss_weights = {
        'global_output':   global_weight,
        'personal_output': personal_weight}
    del loss_fn
    return model


# ── Build one dual-head model from a config dict ──────────────────────────────
def _build_dual_head_model(input_shape, n_class, cfg):

    t_in, t_out  = shared_backbone(input_shape)
    x_flat       = Flatten(name='bb_flatten')(t_out)

    # Global head — always the same structure
    global_out   = Dense(n_class, activation='softmax', name='global_output')(x_flat)

    # Personal head — varies by config
    x_pers = Dense(cfg['d1'], activation='gelu',
                   kernel_regularizer=tf.keras.regularizers.l2(0.01),
                   name='personal_dense_1')(x_flat)
    if cfg['d2'] is not None:
        x_pers = Dense(cfg['d2'], activation='relu',
                       kernel_regularizer=tf.keras.regularizers.l2(0.05),
                       name='personal_dense_2')(x_pers)

    personal_out = Dense(n_class, activation='softmax', name='personal_output')(x_pers)

    model = Model(inputs=t_in,
                  outputs={'global_output':   global_out,
                           'personal_output': personal_out})
    return model


# ── Initial dual-head creation ────────────────────────────────────────────────
def _create_dual_head_model(input_shape, n_class, global_model):

    cfg   = HEAD_CONFIGS[0]  
    model = None

    try:
        gc.collect()
        model = _build_dual_head_model(input_shape, n_class, cfg)
        sync_full_model_weights(model, global_model)
        _compile_dual_head_model(model)
        print(f"      ✓ Dual-head initialized: head={_arch_str(cfg)}"
              f" [trials will explore larger heads over rounds]")

    except Exception as e:
        print(f"      Head creation failed: {e}")
        if model is not None:
            safe_model_delete(model)
            model = None
        gc.collect()

        # Fallback: retry same smallest head
        print("   Retrying fallback head (64,)...")
        try:
            model = _build_dual_head_model(input_shape, n_class, cfg)
            sync_full_model_weights(model, global_model)
            _compile_dual_head_model(model)
            print("      ✓ Fallback created successfully")
        except Exception as e2:
            raise RuntimeError(
                f"Cannot create dual-head model: {e2}") from e2

    return model


# User model creation
def create_model_for_user(user_id, input_shape, n_class, severity,
                           global_model, use_ewc=True, severity_aware=True):

    print(f"\n    Creating model for {user_id} [{severity}] "
          f"[EWC={'ON' if use_ewc else 'OFF'}] "
          f"[SevAware={'ON' if severity_aware else 'OFF'}]...")

    if not severity_aware:
        # ABLATION: Every client gets dual-head regardless of severity
        model = _create_dual_head_model(input_shape, n_class, global_model)
        print(f"      ✓ Dual-head [ABLATION — all personalized, "
              f"severity={severity} ignored]")

    elif use_ewc and severity in PERSONAL_SEVERITIES:
        # Original: Only Stable gets dual-head
        model = _create_dual_head_model(input_shape, n_class, global_model)

    else:
        # Single-head for: all ablation clients OR high-severity with EWC
        shared_input, shared_output = shared_backbone(input_shape)
        x          = Flatten(name='bb_flatten')(shared_output)
        global_out = Dense(n_class, activation='softmax',
                           name='global_output')(x)
        model      = Model(inputs=shared_input, outputs=global_out)
        sync_full_model_weights(model, global_model)
        model.compile(
            optimizer=tf.keras.optimizers.RMSprop(0.001),
            loss=CategoricalCrossentropy(label_smoothing=0.1),
            metrics=['accuracy'])

        if not use_ewc:
            print(f"  Single-head [ABLATION — no personalization]")
        else:
            print(f" Single-head [{severity}]")

    gc.collect()
    return model


In [ ]:
# Trial for best personalized layers
def should_run_trial(round_num, user_id, client_severities,
                     trial_counts,
                     trial_interval=5,
                     max_trials=6,
                     use_ewc=True):

    if not use_ewc:
        return False         
    if round_num == 0:
        return False
    if round_num % trial_interval != 0:
        return False
    if client_severities.get(user_id, 'HIGH') not in PERSONAL_SEVERITIES:
        return False
    return trial_counts.get(user_id, 0) < max_trials


# Periodic head re-evaluation 
def run_lightweight_trial(user_id, current_model, splits, global_model,
                          input_shape, n_class, severity,
                          trial_counts,
                          num_trials=3,
                          quick_epochs=2,
                          batch_size=32):

    current_backbone      = get_backbone_weights(current_model)
    current_global_output = get_global_output_weights(current_model)
    current_best_acc = evaluate_user_model(
        current_model,
        splits['val_data'],
        splits['val_labels'],
        severity, batch_size)

    start_idx = trial_counts.get(user_id, 0)
    end_idx   = min(start_idx + num_trials, len(HEAD_CONFIGS))
    configs   = HEAD_CONFIGS[start_idx:end_idx]

    if not configs:
        print(f"    {user_id}: all head configs already explored")
        del current_backbone
        if current_global_output is not None:
            del current_global_output
        gc.collect()
        return current_model, False

    best_trial_model = None
    best_trial_acc   = current_best_acc
    print(f"    {user_id}: head trial "
          f"[configs {start_idx+1}–{end_idx} of {len(HEAD_CONFIGS)}] "
          f"(baseline val_acc={current_best_acc:.4f})")

    for idx, cfg in enumerate(configs):
        trial_model = None
        arch        = _arch_str(cfg)   
        try:

            trial_model = _build_dual_head_model(input_shape, n_class, cfg)
            set_backbone_weights(trial_model, current_backbone)
            if current_global_output:
                set_global_output_weights(trial_model, current_global_output)


            for layer in trial_model.layers:
                if any(bb in layer.name for bb in BACKBONE_LAYER_NAMES):
                    layer.trainable = False

            _compile_dual_head_model(trial_model)

            train_gen = FedPerDualHeadGenerator(
                splits['train_data'], splits['train_labels'],
                batch_size=batch_size, shuffle=True)
            trial_model.fit(train_gen, epochs=quick_epochs, verbose=0)
            del train_gen
            gc.collect()

            # Unfreeze all before evaluation
            for layer in trial_model.layers:
                layer.trainable = True
            _compile_dual_head_model(trial_model)

            # Evaluate on val data only
            acc     = evaluate_user_model(
                trial_model,
                splits['val_data'],
                splits['val_labels'],
                severity, batch_size)
            is_best = acc > best_trial_acc
            print(f"      T{start_idx + idx + 1}: head={arch}"
                  f" → val_acc={acc:.4f}"
                  f"{'  ← BEST' if is_best else ''}")

            if is_best:
                if best_trial_model is not None:
                    safe_model_delete(best_trial_model)
                    best_trial_model = None
                best_trial_acc   = acc
                best_trial_model = trial_model
                trial_model      = None
            else:
                safe_model_delete(trial_model)
                trial_model = None
                gc.collect()

        except Exception as e:
            print(f"      T{start_idx + idx + 1} ({arch}) failed: {e}")
            if trial_model is not None:
                safe_model_delete(trial_model)
                trial_model = None
            gc.collect()

    del current_backbone
    if current_global_output is not None:
        del current_global_output
    gc.collect()

    if best_trial_model is not None and best_trial_acc > current_best_acc:
        print(f"    {user_id}: ✓ head updated "
              f"(+{best_trial_acc - current_best_acc:.4f})")
        return best_trial_model, True
    else:
        if best_trial_model is not None:
            safe_model_delete(best_trial_model)
            best_trial_model = None
        print(f"    {user_id}: no improvement, keeping current head")
        return current_model, False


# Evaluate user model 
def evaluate_user_model(model, data, labels, client_severity,
                        batch_size=32, global_model=None,
                        use_ewc=True):

    n_samples = len(data)
    n_batches = max(1, (n_samples + batch_size - 1) // batch_size)

    # Determine which model to use
    use_global = (use_ewc and
                  client_severity in GLOBAL_SEVERITIES and
                  global_model is not None)
    eval_model = global_model if use_global else model

    all_preds  = []
    all_labels = []

    try:
        for batch_idx in range(n_batches):
            start = batch_idx * batch_size
            end   = min(start + batch_size, n_samples)

            batch_x = data[start:end]
            batch_y = labels[start:end]

            # Direct call — no cuDNN profiling
            preds = eval_model(batch_x, training=False)

            # Pick correct output
            if isinstance(preds, dict):
                if use_ewc and client_severity in PERSONAL_SEVERITIES:
                    preds = preds.get('personal_output',
                                      preds.get('global_output'))
                else:
                    preds = preds['global_output']
            elif isinstance(preds, (list, tuple)):
                out_names = eval_model.output_names
                if (use_ewc and
                        client_severity in PERSONAL_SEVERITIES and
                        'personal_output' in out_names):
                    preds = preds[out_names.index('personal_output')]
                else:
                    idx = (out_names.index('global_output')
                           if 'global_output' in out_names else 0)
                    preds = preds[idx]

            all_preds.append(np.argmax(preds.numpy(), axis=1))
            all_labels.append(np.argmax(batch_y, axis=1))

            del batch_x, batch_y, preds

        all_preds  = np.concatenate(all_preds)
        all_labels = np.concatenate(all_labels)
        accuracy   = float(np.mean(all_preds == all_labels))

        del all_preds, all_labels
        gc.collect()
        return accuracy

    except tf.errors.ResourceExhaustedError as e:
        print(f"  GPU OOM in evaluate_user_model: {e}")
        gc.collect()
        return 0.0
    except Exception as e:
        print(f"    Eval error: {e}")
        gc.collect()
        return 0.0

In [ ]:
# FedProx for G1 clients
def train_local_fedprox_unified_v2(local_model, global_backbone_weights,
                                    train_data, train_labels,
                                    mu, epochs, batch_size,
                                    val_data=None, val_labels=None,
                                    client_severity='LOW',
                                    global_output_ref=None,
                                    severity_metrics=None,
                                    round_num=0, 
                                    total_rounds=50,
                                    use_ewc=True):

    optimizer    = local_model.optimizer
    is_dual_head = (hasattr(local_model, 'output_names') and
                    len(local_model.output_names) > 1)

    if is_dual_head:
        # Loss functions 
        if hasattr(local_model, '_custom_loss_fns'):
            global_loss_fn   = local_model._custom_loss_fns['global_output']
            personal_loss_fn = local_model._custom_loss_fns['personal_output']
        else:
            global_loss_fn   = local_model.loss.get('global_output',   local_model.loss)
            personal_loss_fn = local_model.loss.get('personal_output', local_model.loss)

        if hasattr(local_model, '_custom_loss_weights'):
            w_global   = local_model._custom_loss_weights['global_output']
            w_personal = local_model._custom_loss_weights['personal_output']
        else:
            w_global, w_personal = 0.3, 0.7

        # Backbone variable IDs 
        backbone_var_ids = set()
        for layer_name in BACKBONE_LAYER_NAMES:
            try:
                layer = local_model.get_layer(layer_name)
                for var in layer.trainable_variables:
                    backbone_var_ids.add(id(var))
            except ValueError:
                continue

        # Backbone proximal references (TF constants) 
        backbone_refs = {}
        for layer_name in BACKBONE_LAYER_NAMES:
            try:
                layer = local_model.get_layer(layer_name)
                if layer_name in global_backbone_weights:
                    for var_idx, var in enumerate(layer.trainable_variables):
                        if var_idx < len(global_backbone_weights[layer_name]):
                            backbone_refs[id(var)] = tf.constant(
                                global_backbone_weights[layer_name][var_idx],
                                dtype=tf.float32)
            except ValueError:
                continue

        mu_half = tf.constant(mu / 2.0, dtype=tf.float32)

        history = {'train_accuracy': [], 'val_accuracy': [],
                   'prox_loss': [], 'ce_loss': [], 'total_loss': [],
                   'output_prox_strength': 0.0}

        n_samples = len(train_data)
        n_batches = max(1, n_samples // batch_size)

        for epoch in range(epochs):
            indices    = np.random.permutation(n_samples)
            epoch_ce, epoch_prox, epoch_acc = [], [], []

            for batch_idx in range(n_batches):
                start   = batch_idx * batch_size
                end     = min(start + batch_size, n_samples)
                batch_x = train_data[indices[start:end]]
                batch_y = train_labels[indices[start:end]]

                with tf.GradientTape() as tape:
                    predictions = local_model(batch_x, training=True)

                    if isinstance(predictions, dict):
                        global_pred   = predictions['global_output']
                        personal_pred = predictions['personal_output']
                    elif isinstance(predictions, (list, tuple)):
                        output_names  = local_model.output_names
                        idx_g         = output_names.index('global_output')
                        idx_p         = output_names.index('personal_output')
                        global_pred   = predictions[idx_g]
                        personal_pred = predictions[idx_p]
                    else:
                        global_pred   = predictions
                        personal_pred = predictions

                    ce_global   = global_loss_fn(batch_y, global_pred)
                    ce_personal = personal_loss_fn(batch_y, personal_pred)
                    ce_loss     = w_global * ce_global + w_personal * ce_personal
                    pred_cls    = tf.argmax(personal_pred, axis=1)

                    if local_model.losses:
                        ce_loss = ce_loss + tf.add_n(local_model.losses)

                    prox_term = tf.constant(0.0, dtype=tf.float32)
                    for var in local_model.trainable_weights:
                        if id(var) in backbone_refs:
                            prox_term += tf.reduce_sum(
                                tf.square(var - backbone_refs[id(var)]))
                    prox_loss  = mu_half * prox_term
                    total_loss = ce_loss + prox_loss

                all_gradients = tape.gradient(
                    total_loss, local_model.trainable_weights)

                backbone_grads_vars = []
                personal_grads_vars = []
                for grad, var in zip(all_gradients, local_model.trainable_weights):
                    if grad is None:
                        continue
                    if id(var) in backbone_var_ids:
                        backbone_grads_vars.append((grad, var))
                    else:
                        personal_grads_vars.append((grad, var))

                if backbone_grads_vars:
                    bb_grads, bb_vars = zip(*backbone_grads_vars)
                    bb_grads_clipped, _ = tf.clip_by_global_norm(
                        list(bb_grads), clip_norm=1.0)
                    optimizer.apply_gradients(zip(bb_grads_clipped, bb_vars))

                if personal_grads_vars:
                    p_grads, p_vars = zip(*personal_grads_vars)
                    p_grads_clipped, _ = tf.clip_by_global_norm(
                        list(p_grads), clip_norm=5.0)
                    optimizer.apply_gradients(zip(p_grads_clipped, p_vars))

                epoch_ce.append(float(ce_loss))
                epoch_prox.append(float(prox_loss))
                true_cls = tf.argmax(batch_y, axis=1)
                epoch_acc.append(float(tf.reduce_mean(
                    tf.cast(tf.equal(pred_cls, true_cls), tf.float32))))

            history['train_accuracy'].append(np.mean(epoch_acc))
            history['ce_loss'].append(np.mean(epoch_ce))
            history['prox_loss'].append(np.mean(epoch_prox))
            history['total_loss'].append(
                np.mean(epoch_ce) + np.mean(epoch_prox))

            if val_data is not None and val_labels is not None:
                val_acc = evaluate_user_model(
                    local_model, val_data, val_labels,
                    client_severity, batch_size,
                    use_ewc=use_ewc)
                history['val_accuracy'].append(val_acc)

        del backbone_refs
        del mu_half
        gc.collect()

        return history

    else:
        loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05)
        all_weights_ref = [tf.constant(w.numpy(), dtype=tf.float32)
                           for w in local_model.trainable_weights]

        mu_half = tf.constant(mu / 2.0, dtype=tf.float32)

        history = {'train_accuracy': [], 'val_accuracy': [],
                   'prox_loss': [], 'ce_loss': [], 'total_loss': [],
                   'output_prox_strength': 0.0}

        n_samples = len(train_data)
        n_batches = max(1, n_samples // batch_size)

        for epoch in range(epochs):
            indices    = np.random.permutation(n_samples)
            epoch_ce, epoch_prox, epoch_acc = [], [], []

            for batch_idx in range(n_batches):
                start   = batch_idx * batch_size
                end     = min(start + batch_size, n_samples)
                batch_x = train_data[indices[start:end]]
                batch_y = train_labels[indices[start:end]]

                with tf.GradientTape() as tape:
                    predictions = local_model(batch_x, training=True)
                    ce_loss     = loss_fn(batch_y, predictions)

                    prox_term = tf.constant(0.0, dtype=tf.float32)
                    for local_w, global_w in zip(
                            local_model.trainable_weights, all_weights_ref):
                        prox_term += tf.reduce_sum(
                            tf.square(local_w - global_w))
                    prox_loss  = mu_half * prox_term
                    total_loss = ce_loss + prox_loss

                gradients = tape.gradient(
                    total_loss, local_model.trainable_weights)
                optimizer.apply_gradients(
                    zip(gradients, local_model.trainable_weights))

                epoch_ce.append(float(ce_loss))
                epoch_prox.append(float(prox_loss))
                pred_cls = tf.argmax(predictions, axis=1)
                true_cls = tf.argmax(batch_y, axis=1)
                epoch_acc.append(float(tf.reduce_mean(
                    tf.cast(tf.equal(pred_cls, true_cls), tf.float32))))

            history['train_accuracy'].append(np.mean(epoch_acc))
            history['ce_loss'].append(np.mean(epoch_ce))
            history['prox_loss'].append(np.mean(epoch_prox))
            history['total_loss'].append(
                np.mean(epoch_ce) + np.mean(epoch_prox))

            if val_data is not None and val_labels is not None:
                val_acc = evaluate_user_model(
                    local_model, val_data, val_labels,
                    client_severity, batch_size,
                    use_ewc=use_ewc)
                history['val_accuracy'].append(val_acc)

        del all_weights_ref
        del mu_half
        del loss_fn
        gc.collect()

        return history

In [ ]:
# Federated Functions
def compute_quality_weights(user_ids, per_user_splits, severity_metrics):

    raw_weights = {}
    for uid in user_ids:
        data_size     = len(per_user_splits[uid]['train_data'])
        overlap_ratio = severity_metrics[uid]['overlap_ratio']
        raw_weights[uid] = data_size * (0.3 + 0.7 * overlap_ratio)

    total      = sum(raw_weights.values())
    normalized = {uid: w / total for uid, w in raw_weights.items()}
    return normalized


def federated_avg_backbone(client_models, client_weights, verbose=False):
    all_backbones = {}

    for uid, model in client_models.items():
        weight   = client_weights.get(uid, 1.0 / len(client_models))
        backbone = get_backbone_weights(model)
        for layer_name, layer_weights in backbone.items():
            if layer_name not in all_backbones:
                all_backbones[layer_name] = []
            all_backbones[layer_name].append((layer_weights, weight, uid))

    averaged_backbone = {}

    for layer_name in list(all_backbones.keys()):   
        contributions = all_backbones[layer_name]
        if not contributions:
            continue

        ref_shapes = [w.shape for w in contributions[0][0]]
        compatible = [
            (ws, wt, uid) for ws, wt, uid in contributions
            if [w.shape for w in ws] == ref_shapes
        ]

        if not compatible:
            if verbose:
                print(f"   {layer_name}: no compatible weights")
            del all_backbones[layer_name]            
            continue

        total_weight = sum(wt for _, wt, _ in compatible)
        n_arrays     = len(compatible[0][0])

        averaged_layer = []
        for arr_idx in range(n_arrays):
            weighted_sum = np.zeros_like(compatible[0][0][arr_idx])
            for layer_ws, user_wt, _ in compatible:
                weighted_sum += layer_ws[arr_idx] * (user_wt / total_weight)
            averaged_layer.append(weighted_sum)

        averaged_backbone[layer_name] = averaged_layer

        del compatible
        del all_backbones[layer_name]
        gc.collect()

        if verbose:
            print(f"   {layer_name}: averaged")

    del all_backbones
    gc.collect()

    return averaged_backbone


def federated_avg_global_output(client_models, client_weights, verbose=False):
    contributions = []
    weights_list  = []
    user_ids      = []

    for uid, model in client_models.items():
        ws = get_global_output_weights(model)
        if ws is not None:
            contributions.append(ws)
            weights_list.append(
                client_weights.get(uid, 1.0 / len(client_models)))
            user_ids.append(uid)

    if not contributions:
        if verbose:
            print(f"   global_output: no contributions")
        return None

    total_w = sum(weights_list)
    if total_w == 0:
        return None

    ref_shapes = [w.shape for w in contributions[0]]
    compatible = [
        (ws, wt, uid)
        for ws, wt, uid in zip(contributions, weights_list, user_ids)
        if [w.shape for w in ws] == ref_shapes
    ]

    if not compatible:
        if verbose:
            print(f"   global_output: no compatible weights")
        del contributions, weights_list, user_ids  
        gc.collect()
        return None

    total_w = sum(wt for _, wt, _ in compatible)

    avg = []
    for i in range(len(compatible[0][0])):
        weighted_sum = np.zeros_like(compatible[0][0][i])
        for ws, uw, _ in compatible:
            weighted_sum += ws[i] * (uw / total_w)
        avg.append(weighted_sum)

    del compatible
    del contributions
    del weights_list
    del user_ids
    gc.collect()

    if verbose:
        print(f"   global_output: averaged from {len(avg)} arrays")

    return avg

In [ ]:
# EWC IMPLEMENTATION FOR NEW CLIENT ADAPTATION
class EWCCache:
    def __init__(self):
        self._cache            = {}
        self._lambda           = None
        self._layer_to_weights = None
        self._trainable_map    = {}   
        self._model_id         = None 

    def build(self, reference_weights_numpy, fisher_numpy,
              layer_to_weights, lambda_ewc):
        self._cache.clear()
        self._trainable_map.clear()   
        self._model_id         = None 
        self._lambda           = tf.constant(lambda_ewc, dtype=tf.float32)
        self._layer_to_weights = layer_to_weights

        for layer_idx, wi_list in layer_to_weights.items():
            for wi in wi_list:
                if (fisher_numpy[wi] is not None and
                        reference_weights_numpy[wi] is not None):
                    self._cache[wi] = (
                        tf.constant(reference_weights_numpy[wi],
                                    dtype=tf.float32),
                        tf.constant(fisher_numpy[wi],
                                    dtype=tf.float32)
                    )
        print(f"  EWCCache built: {len(self._cache)} weight arrays cached")
        return self

    def refresh(self, reference_weights_numpy, fisher_numpy, lambda_ewc):
        self._cache.clear()
        self._trainable_map.clear()   
        self._model_id = None
        gc.collect()
        return self.build(reference_weights_numpy, fisher_numpy,
                          self._layer_to_weights, lambda_ewc)

    def is_ready(self):
        return len(self._cache) > 0

    def _get_trainable_map(self, model):
        if id(model) == self._model_id and self._trainable_map:
            return self._trainable_map  

        trainable_map = {}
        global_idx    = 0
        t_idx         = 0
        for layer in model.layers:
            for _ in layer.get_weights():
                if layer.trainable:
                    trainable_map[global_idx] = \
                        model.trainable_weights[t_idx]
                    t_idx += 1
                global_idx += 1

        self._trainable_map = trainable_map
        self._model_id      = id(model)
        return self._trainable_map

    def compute_loss(self, model):
        if not self.is_ready():
            return tf.constant(0.0, dtype=tf.float32)

        trainable_map = self._get_trainable_map(model) 
        ewc_loss      = tf.constant(0.0, dtype=tf.float32)

        for wi, (ref_tf, fisher_tf) in self._cache.items():
            if wi not in trainable_map:
                continue
            current_w = trainable_map[wi]
            if current_w.shape == ref_tf.shape == fisher_tf.shape:
                diff     = current_w - ref_tf
                penalty  = tf.reduce_sum(fisher_tf * tf.square(diff))
                ewc_loss = ewc_loss + penalty

        return self._lambda * ewc_loss


def compute_ewc_loss_numpy_refs(model, reference_weights_numpy,
                                 fisher_numpy, layer_to_weights,
                                 lambda_ewc, ewc_cache=None):
    if ewc_cache is not None and ewc_cache.is_ready():
        return ewc_cache.compute_loss(model)

    # Fallback path
    ewc_loss   = tf.constant(0.0, dtype=tf.float32)
    trainable_map = {}
    global_idx    = 0
    t_idx         = 0
    for layer in model.layers:
        for _ in layer.get_weights():
            if layer.trainable:
                trainable_map[global_idx] = model.trainable_weights[t_idx]
                t_idx += 1
            global_idx += 1

    for layer_idx, wi_list in layer_to_weights.items():
        for wi in wi_list:
            if (wi not in trainable_map or
                    fisher_numpy[wi] is None or
                    reference_weights_numpy[wi] is None):
                continue
            current_w   = trainable_map[wi]
            reference_w = tf.constant(reference_weights_numpy[wi],
                                      dtype=tf.float32)
            fisher      = tf.constant(fisher_numpy[wi], dtype=tf.float32)
            if current_w.shape == reference_w.shape == fisher.shape:
                diff     = current_w - reference_w
                penalty  = tf.reduce_sum(fisher * tf.square(diff))
                ewc_loss = ewc_loss + penalty
            del reference_w, fisher  

    del trainable_map                
    return lambda_ewc * ewc_loss



# Local Fisher Computation (runs on each client)
def compute_local_fisher_numpy(model, train_data, train_labels,
                                layer_to_weights, batch_size=32,
                                sample_ratio=0.3, user_id=None):

    # Sample local data
    n_samples = max(1, int(len(train_data) * sample_ratio))
    idx = np.random.choice(len(train_data), n_samples, replace=False)
    sampled_data   = train_data[idx]
    sampled_labels = train_labels[idx]
    
    if user_id:
        print(f"    {user_id}: computing local Fisher "
              f"({n_samples}/{len(train_data)} samples)")
    
    # Initialize local Fisher (same shape as global weights)
    all_weights  = model.get_weights()
    total_w      = len(all_weights)
    local_fisher = [None] * total_w
    
    for layer_idx, wi_list in layer_to_weights.items():
        for wi in wi_list:
            local_fisher[wi] = np.zeros_like(
                all_weights[wi], dtype=np.float32)
    del all_weights
    
    # Build trainable weight index map
    trainable_to_global = {}
    global_idx = 0
    t_idx      = 0
    for layer in model.layers:
        for _ in layer.get_weights():
            if layer.trainable:
                trainable_to_global[t_idx] = global_idx
                t_idx += 1
            global_idx += 1
    global_to_trainable = {v: k for k, v in trainable_to_global.items()}
    del trainable_to_global
    
    # Accumulate squared gradients
    n_batches = max(1, (len(sampled_data) + batch_size - 1) // batch_size)
    
    for batch_idx in range(n_batches):
        start   = batch_idx * batch_size
        end     = min(start + batch_size, len(sampled_data))
        batch_x = tf.constant(sampled_data[start:end], dtype=tf.float32)
        batch_y = tf.constant(sampled_labels[start:end], dtype=tf.float32)
        
        with tf.GradientTape() as tape:
            preds = model(batch_x, training=False)
            loss  = tf.reduce_mean(
                tf.keras.losses.categorical_crossentropy(batch_y, preds))
        
        grads = tape.gradient(loss, model.trainable_weights)
        
        for layer_idx, wi_list in layer_to_weights.items():
            for wi in wi_list:
                t_i = global_to_trainable.get(wi)
                if t_i is not None and grads[t_i] is not None:
                    local_fisher[wi] += grads[t_i].numpy() ** 2
        
        del tape, preds, loss, grads, batch_x, batch_y
    
    # Normalize by number of batches
    for layer_idx, wi_list in layer_to_weights.items():
        for wi in wi_list:
            if local_fisher[wi] is not None:
                local_fisher[wi] /= float(n_batches)
    
    del global_to_trainable
    del sampled_data, sampled_labels
    gc.collect()
    
    return local_fisher, n_samples


# Server-Side Federated Aggregation
def aggregate_fisher_federated(client_fishers, layer_to_weights,
                                 aggregation='weighted',
                                 severity_metrics=None,
                                 verbose=True):

    if not client_fishers:
        return None
    
    # ── Compute aggregation weights ───────────────────────────────────
    if aggregation == 'weighted':
        # Weight by number of samples used for Fisher computation
        total_samples = sum(n for _, n in client_fishers.values())
        agg_weights = {
            uid: n / total_samples
            for uid, (_, n) in client_fishers.items()
        }
        
    elif aggregation == 'quality_weighted':
        # Weight by samples × overlap_ratio (same as FedAvg backbone)
        raw_weights = {}
        for uid, (_, n_samples) in client_fishers.items():
            overlap = (severity_metrics[uid]['overlap_ratio']
                       if severity_metrics and uid in severity_metrics
                       else 1.0)
            raw_weights[uid] = n_samples * (0.3 + 0.7 * overlap)
        total = sum(raw_weights.values())
        agg_weights = {uid: w / total for uid, w in raw_weights.items()}
        
    else:  # equal
        n_clients   = len(client_fishers)
        agg_weights = {uid: 1.0 / n_clients for uid in client_fishers}
    
    if verbose:
        print(f"\n    Federated Fisher Aggregation ({aggregation}):")
        for uid, w in sorted(agg_weights.items()):
            _, n = client_fishers[uid]
            print(f"      {uid}: weight={w:.4f} (n_samples={n})")
    
    # ── Get reference structure ───────────────────────────────────────
    ref_fisher = next(iter(client_fishers.values()))[0]
    total_w    = len(ref_fisher)
    
    aggregated_fisher = [None] * total_w
    for layer_idx, wi_list in layer_to_weights.items():
        for wi in wi_list:
            aggregated_fisher[wi] = np.zeros_like(
                ref_fisher[wi], dtype=np.float32)
    
    # ── Weighted sum ──────────────────────────────────────────────────
    for uid, (local_fisher, _) in client_fishers.items():
        w = agg_weights[uid]
        for layer_idx, wi_list in layer_to_weights.items():
            for wi in wi_list:
                if (local_fisher[wi] is not None and
                        aggregated_fisher[wi] is not None):
                    # Shape safety check
                    if local_fisher[wi].shape == aggregated_fisher[wi].shape:
                        aggregated_fisher[wi] += w * local_fisher[wi]
    
    if verbose:
        n_aggregated = sum(
            1 for layer_idx, wi_list in layer_to_weights.items()
            for wi in wi_list
            if aggregated_fisher[wi] is not None)
        print(f"    ✓ Aggregated {n_aggregated} weight arrays "
              f"from {len(client_fishers)} clients")
    
    return aggregated_fisher


def create_fisher_federated(model, per_user_splits, shared_layer_indices,
                              batch_size=32, sample_ratio=0.3,
                              aggregation='quality_weighted',
                              severity_metrics=None,
                              verbose=True):
    
    print(f"\n  Computing Federated Fisher Information...")
    print(f"  Clients: {sorted(per_user_splits.keys())}")
    print(f"  Aggregation: {aggregation}")
    print(f"  {'─'*55}")
    
    layer_to_weights = {}
    weight_idx = 0
    for layer_idx, layer in enumerate(model.layers):
        if layer.trainable and layer.get_weights():
            num_w = len(layer.get_weights())
            if layer_idx in shared_layer_indices:
                layer_to_weights[layer_idx] = list(
                    range(weight_idx, weight_idx + num_w))
            weight_idx += num_w
    
    print(f"  Shared layers for Fisher: "
          f"{len(layer_to_weights)} "
          f"(indices: {list(layer_to_weights.keys())})")
    
    # Each client computes LOCAL Fisher
    client_fishers = {}
    
    for user_id in sorted(per_user_splits.keys()):
        splits = per_user_splits[user_id]
        try:
            local_fisher, n_used = compute_local_fisher_numpy(
                model=model,
                train_data=splits['train_data'],
                train_labels=splits['train_labels'],
                layer_to_weights=layer_to_weights,
                batch_size=batch_size,
                sample_ratio=sample_ratio,
                user_id=user_id)
            
            client_fishers[user_id] = (local_fisher, n_used)
            
        except Exception as e:
            print(f"    {user_id}: Fisher computation failed — {e}")
            import traceback; traceback.print_exc()
        finally:
            gc.collect()
    
    print(f"\n  Received Fisher from "
          f"{len(client_fishers)}/{len(per_user_splits)} clients")
    
    # Server aggregates 
    aggregated_fisher = aggregate_fisher_federated(
        client_fishers=client_fishers,
        layer_to_weights=layer_to_weights,
        aggregation=aggregation,
        severity_metrics=severity_metrics,
        verbose=verbose)
    
    for uid in list(client_fishers.keys()):
        del client_fishers[uid]
    del client_fishers
    gc.collect()
    
    
    print(f"\n  Federated Fisher computed for "
          f"{len(layer_to_weights)} layers")
    print(f"{'='*60}\n")
    
    return aggregated_fisher, layer_to_weights

# Federated Fisher Refresh
def refresh_fisher_federated(model, per_user_splits, shared_layer_indices,
                               old_fisher_numpy, layer_to_weights,
                               batch_size=32, sample_ratio=0.3,
                               ema_alpha=0.3,
                               aggregation='quality_weighted',
                               severity_metrics=None):

    print(f"\n  Federated Fisher Refresh (EMA α={ema_alpha})")
    
    new_fisher, _ = create_fisher_federated(
        model=model,
        per_user_splits=per_user_splits,
        shared_layer_indices=shared_layer_indices,
        batch_size=batch_size,
        sample_ratio=sample_ratio,
        aggregation=aggregation,
        severity_metrics=severity_metrics,
        verbose=False)
    
    # EMA update (reuse your existing function)
    updated_fisher = update_fisher_numpy_ema(
        old_fisher_numpy=old_fisher_numpy,
        new_fisher_numpy=new_fisher,
        layer_to_weights=layer_to_weights,
        alpha=ema_alpha)
    
    del new_fisher
    gc.collect()
    
    print(f"  ✓ Federated Fisher refreshed (EMA)")
    return updated_fisher

def update_fisher_numpy_ema(old_fisher_numpy, new_fisher_numpy,
                             layer_to_weights, alpha=0.3):
    for layer_idx, wi_list in layer_to_weights.items():
        for wi in wi_list:
            if (old_fisher_numpy[wi] is not None and
                    new_fisher_numpy[wi] is not None):
                old_fisher_numpy[wi] *= (1 - alpha)
                old_fisher_numpy[wi] += alpha * new_fisher_numpy[wi]
                new_fisher_numpy[wi]  = None 
    gc.collect()
    return old_fisher_numpy

In [ ]:
# EWC for G2 clients
def train_local_ewc_only(
        local_model,
        global_backbone_weights,
        train_data,
        train_labels,
        epochs,
        batch_size,
        val_data=None,
        val_labels=None,
        client_severity='LOW',
        reference_weights_numpy=None,
        fisher_numpy=None,
        layer_to_weights=None,
        lambda_ewc=0.4,
        ewc_cache=None,
        use_ewc=True):

    optimizer    = local_model.optimizer
    is_dual_head = (hasattr(local_model, 'output_names') and
                    len(local_model.output_names) > 1)

    # EWC active check 
    ewc_is_active = (
        use_ewc                             and
        layer_to_weights        is not None and
        fisher_numpy            is not None and
        reference_weights_numpy is not None and
        lambda_ewc > 0                      and
        any(fisher_numpy[wi] is not None
            for wi_list in layer_to_weights.values()
            for wi in wi_list))

    if use_ewc and not ewc_is_active:
        print(f" EWC requested but Fisher/reference not ready!")

    ce_loss_fn = tf.keras.losses.CategoricalCrossentropy(
                     label_smoothing=0.05)

    frozen_layers = []   # personal layers frozen in ablation

    # Branch selection 
    if use_ewc and is_dual_head:
        # WITH EWC + DUAL → weighted CE on both + EWC
        use_dual_ce = True
        print(f"    [EWC OPEN  | DUAL  ] "
              f"CE(0.3*global+0.7*personal) + EWC")

    elif use_ewc and not is_dual_head:
        # WITH EWC + SINGLE → CE on global + EWC
        use_dual_ce = False
        print(f"    [EWC OPEN  | SINGLE] CE on global_output + EWC")

    elif not use_ewc and is_dual_head:
        # WITHOUT EWC + DUAL → CE on global ONLY (ablation)
        # Freeze personal layers so they are NOT trained
        # Evaluation will also use global_output for this client
        use_dual_ce = False
        for layer in local_model.layers:
            if 'personal' in layer.name:
                layer.trainable = False
                frozen_layers.append(layer.name)
        print(f"    [EWC CLOSED| DUAL  ] "
              f"CE on global_output ONLY (ablation)")
        if frozen_layers:
            print(f"    [FROZEN] {frozen_layers}")

    else:
        # WITHOUT EWC + SINGLE → CE on global ONLY (ablation)
        use_dual_ce = False
        print(f"    [EWC CLOSED| SINGLE] CE on global_output (ablation)")

    
    history = {
        'train_accuracy' : [],
        'val_accuracy'   : [],
        'task_loss'      : [],
        'ewc_loss'       : [],
        'total_loss'     : [],
        'ewc_active'     : ewc_is_active,
        'loss_mode'      : ('CE_dual+EWC'  if (use_ewc and is_dual_head)
                            else 'CE+EWC'  if use_ewc
                            else 'CE_global_ablation')
    }

    n_samples = len(train_data)
    n_batches = max(1, n_samples // batch_size)

    for epoch in range(epochs):
        indices  = np.random.permutation(n_samples)
        epoch_task, epoch_ewc, epoch_acc = [], [], []

        for batch_idx in range(n_batches):
            start   = batch_idx * batch_size
            end     = min(start + batch_size, n_samples)
            batch_x = tf.constant(
                train_data[indices[start:end]],   dtype=tf.float32)
            batch_y = tf.constant(
                train_labels[indices[start:end]], dtype=tf.float32)

            with tf.GradientTape() as tape:
                predictions = local_model(batch_x, training=True)

                # Task loss 
                if use_dual_ce:
                    # WITH EWC + DUAL: weighted CE on both heads
                    if isinstance(predictions, dict):
                        global_pred   = predictions['global_output']
                        personal_pred = predictions['personal_output']
                    elif isinstance(predictions, (list, tuple)):
                        out_names     = local_model.output_names
                        global_pred   = predictions[
                            out_names.index('global_output')]
                        personal_pred = predictions[
                            out_names.index('personal_output')]
                    else:
                        global_pred   = predictions
                        personal_pred = predictions

                    task_loss     = (
                        0.3 * ce_loss_fn(batch_y, global_pred) +
                        0.7 * ce_loss_fn(batch_y, personal_pred))
                    preds_for_acc = personal_pred

                else:
                    # ALL other cases: CE on global_output ONLY
                    # Covers: WITH EWC+SINGLE, WITHOUT EWC+DUAL,
                    #         WITHOUT EWC+SINGLE
                    if isinstance(predictions, dict):
                        pred_out = predictions['global_output']
                    elif isinstance(predictions, (list, tuple)):
                        out_names = local_model.output_names
                        pred_out  = predictions[
                            out_names.index('global_output')]
                    else:
                        pred_out = predictions

                    task_loss     = ce_loss_fn(batch_y, pred_out)
                    preds_for_acc = pred_out

                # L2 regularization
                if local_model.losses:
                    task_loss = task_loss + tf.add_n(local_model.losses)

                # EWC penalty 
                if ewc_is_active:
                    ewc_penalty = compute_ewc_loss_numpy_refs(
                        local_model,
                        reference_weights_numpy,
                        fisher_numpy,
                        layer_to_weights,
                        lambda_ewc,
                        ewc_cache=ewc_cache)
                else:
                    ewc_penalty = tf.constant(0.0, dtype=tf.float32)

                total_loss = task_loss + ewc_penalty

            # Gradient update (frozen layers excluded by TF) 
            trainable_weights = local_model.trainable_weights
            gradients         = tape.gradient(total_loss, trainable_weights)
            clipped, _        = tf.clip_by_global_norm(
                [g if g is not None else tf.zeros_like(v)
                 for g, v in zip(gradients, trainable_weights)],
                clip_norm=1.0)
            optimizer.apply_gradients(zip(clipped, trainable_weights))

            pred_cls = tf.argmax(preds_for_acc, axis=1)
            true_cls = tf.argmax(batch_y,       axis=1)
            epoch_task.append(float(task_loss))
            epoch_ewc.append(float(ewc_penalty))
            epoch_acc.append(float(tf.reduce_mean(
                tf.cast(tf.equal(pred_cls, true_cls), tf.float32))))

            del tape, predictions, preds_for_acc
            del task_loss, ewc_penalty, total_loss
            del gradients, clipped, batch_x, batch_y

        history['train_accuracy'].append(np.mean(epoch_acc))
        history['task_loss'].append(np.mean(epoch_task))
        history['ewc_loss'].append(np.mean(epoch_ewc))
        history['total_loss'].append(
            np.mean(epoch_task) + np.mean(epoch_ewc))

        if val_data is not None and val_labels is not None:
            val_acc = evaluate_user_model(
                local_model, val_data, val_labels,
                client_severity, batch_size,
                use_ewc=use_ewc)         # ← PASS use_ewc HERE
            history['val_accuracy'].append(val_acc)

        gc.collect()

    # Important: unfreeze so FedAvg can read all layer states
    # personal weights unchanged (frozen) → safe to aggregate
    if frozen_layers:
        for layer in local_model.layers:
            if layer.name in frozen_layers:
                layer.trainable = True

    del ce_loss_fn
    gc.collect()
    return history

In [ ]:
# Resource Monitoring
class ResourceMonitor:

    def __init__(self, enable_gpu=False):
        self.process         = psutil.Process()
        self.enable_gpu      = enable_gpu
        self.round_metrics   = defaultdict(dict)
        self.model_cache     = {}
        self._measured_archs = set()
        self.baseline_memory = self.process.memory_info().rss / (1024 * 1024)

        # ── Client model sizes ────────────────────────────────────────
        self.client_model_sizes = {}

        # ── RE tracking ───────────────────────────────────────────────
        # Communication: tracks backbone size sent each round per client
        # Structure: {round_num: {uid: bytes_transmitted}}
        self.communication_log  = defaultdict(dict)

        # Computation: tracks training time per client per round
        # Structure: {round_num: {uid: seconds}}
        self.computation_log    = defaultdict(dict)

        # Accuracy: tracks test accuracy per client per round
        # Structure: {round_num: {uid: float}}
        self.accuracy_log       = defaultdict(dict)

        # Final RE summary (computed at end)
        self.re_summary         = {}

    # ── Existing methods (unchanged) ─────────────────────────────────

    def mark_round_start(self, round_num):
        self.round_metrics[round_num]['start_time']   = time.time()
        self.round_metrics[round_num]['start_memory'] = \
            self.process.memory_info().rss / (1024 * 1024)

    def mark_round_end(self, round_num):
        end_time   = time.time()
        end_memory = self.process.memory_info().rss / (1024 * 1024)
        self.round_metrics[round_num].update({
            'end_time'    : end_time,
            'end_memory'  : end_memory,
            'duration'    : end_time - self.round_metrics[round_num]['start_time'],
            'memory_delta': end_memory - self.round_metrics[round_num]['start_memory'],
            'peak_memory' : max(self.round_metrics[round_num]['start_memory'],
                                end_memory)
        })

    def measure_model_resources(self, model, model_name):
        arch_id = f"{len(model.layers)}_{model.count_params()}"
        if arch_id in self._measured_archs:
            return self.model_cache.get(model_name, {})

        param_count      = model.count_params()
        trainable_params = sum(
            tf.keras.backend.count_params(w)
            for w in model.trainable_weights)

        metrics = {
            'model_name'          : model_name,
            'total_params'        : param_count,
            'trainable_params'    : trainable_params,
            'non_trainable_params': param_count - trainable_params,
            'memory_mb'           : (param_count * 4) / (1024 * 1024),
            'total_layers'        : len(model.layers)
        }
        self.model_cache[model_name]  = metrics
        self._measured_archs.add(arch_id)
        return metrics

    def measure_client_model(self, uid, model, severity,
                              backbone_layer_names=None):
        if backbone_layer_names is None:
            backbone_layer_names = BACKBONE_LAYER_NAMES

        total_params     = model.count_params()
        trainable_params = sum(
            tf.keras.backend.count_params(w)
            for w in model.trainable_weights)

        backbone_params = 0
        for layer in model.layers:
            if any(bb in layer.name for bb in backbone_layer_names):
                backbone_params += layer.count_params()

        head_params  = total_params - backbone_params
        is_dual_head = (hasattr(model, 'output_names') and
                        len(model.output_names) > 1)

        if is_dual_head:
            head_type       = 'DUAL'
            personal_layers = []
            for layer in model.layers:
                if 'personal_dense' in layer.name:
                    units = layer.units if hasattr(layer, 'units') else '?'
                    personal_layers.append(str(units))
            head_config = (f"({','.join(personal_layers)})"
                           if personal_layers else "unknown")
        else:
            head_type   = 'SINGLE'
            head_config = 'global_only'

        memory_mb = (total_params * 4) / (1024 * 1024)

        self.client_model_sizes[uid] = {
            'total_params'    : total_params,
            'trainable_params': trainable_params,
            'backbone_params' : backbone_params,
            'head_params'     : head_params,
            'memory_mb'       : memory_mb,
            'head_type'       : head_type,
            'head_config'     : head_config,
            'severity'        : severity,
        }
        return self.client_model_sizes[uid]

    # RE tracking methods 

    def log_client_communication(self, round_num, uid, model,
                                  backbone_layer_names=None):
        if backbone_layer_names is None:
            backbone_layer_names = BACKBONE_LAYER_NAMES

        backbone_bytes = 0
        for layer in model.layers:
            if any(bb in layer.name for bb in backbone_layer_names):
                backbone_bytes += layer.count_params() * 4  # float32

        self.communication_log[round_num][uid] = backbone_bytes

    def log_client_computation(self, round_num, uid,
                                train_start_time, train_end_time):

        self.computation_log[round_num][uid] = (
            train_end_time - train_start_time)

    def log_client_accuracy(self, round_num, uid, accuracy):
        self.accuracy_log[round_num][uid] = accuracy

    def compute_re_metrics(self, client_severities,
                            group1_user_ids, group2_user_ids):

        if not self.accuracy_log:
            print("   No accuracy logs found. Cannot compute RE.")
            return {}

        all_rounds = sorted(self.accuracy_log.keys())
        final_round = all_rounds[-1]
        all_uids    = list(self.accuracy_log[final_round].keys())

        re_per_client = {}

        for uid in all_uids:
            # ── Final accuracy ────────────────────────────────────────
            final_acc = self.accuracy_log[final_round].get(uid, 0.0)

            # ── Total communication cost (MB) ─────────────────────────
            total_bytes = sum(
                self.communication_log[r].get(uid, 0)
                for r in all_rounds)
            total_mb = total_bytes / (1024 * 1024)

            # ── Total computation time (seconds) ──────────────────────
            total_seconds = sum(
                self.computation_log[r].get(uid, 0.0)
                for r in all_rounds)

            # ── Parameter count ───────────────────────────────────────
            total_params = (
                self.client_model_sizes[uid]['total_params']
                if uid in self.client_model_sizes else 0)
            params_m = total_params / 1e6  # in millions

            # ── RE computations ───────────────────────────────────────
            # Avoid division by zero with small epsilon
            re_comm   = final_acc / max(total_mb,      1e-9)
            re_comp   = final_acc / max(total_seconds,  1e-9)
            re_param  = final_acc / max(params_m,       1e-9)

            re_per_client[uid] = {
                'final_accuracy'   : final_acc,
                'total_comm_mb'    : total_mb,
                'total_comp_sec'   : total_seconds,
                'total_params_m'   : params_m,
                're_communication' : re_comm,
                're_computation'   : re_comp,
                're_parameter'     : re_param,
                'group'            : 'G1' if uid in group1_user_ids else 'G2',
                'severity'         : client_severities.get(uid, 'UNKNOWN'),
                'head_type'        : (self.client_model_sizes[uid]['head_type']
                                      if uid in self.client_model_sizes
                                      else 'UNKNOWN'),
            }

        self.re_summary = re_per_client
        return re_per_client

    def print_re_metrics(self):
        """Print formatted RE metrics table."""
        if not self.re_summary:
            print("    RE metrics not computed yet. "
                  "Call compute_re_metrics() first.")
            return

        print(f"\n  RESOURCE EFFICIENCY (RE) METRICS")
        print(f"  {'User':<10} {'Grp':>4} {'Sev':<9} {'Head':<6} "
              f"{'Acc':>6} {'CommMB':>8} {'CompSec':>9} "
              f"{'RE_Comm':>9} {'RE_Comp':>9} {'RE_Param':>9}")

        for uid in sorted(self.re_summary.keys()):
            r = self.re_summary[uid]
            print(f"  {uid:<10} {r['group']:>4} {r['severity']:<9} "
                  f"{r['head_type']:<6} "
                  f"{r['final_accuracy']:>6.4f} "
                  f"{r['total_comm_mb']:>8.2f} "
                  f"{r['total_comp_sec']:>9.2f} "
                  f"{r['re_communication']:>9.4f} "
                  f"{r['re_computation']:>9.4f} "
                  f"{r['re_parameter']:>9.4f}")


        print(f"\n  By Group:")
        print(f"  {'Group':<6} {'N':>3} "
              f"{'Acc':>7} {'CommMB':>9} {'CompSec':>10} "
              f"{'RE_Comm':>10} {'RE_Comp':>10} {'RE_Param':>10}")
        print(f"  {'─'*70}")

        for grp in ['G1', 'G2']:
            grp_data = [r for r in self.re_summary.values()
                        if r['group'] == grp]
            if grp_data:
                print(f"  {grp:<6} {len(grp_data):>3} "
                      f"{np.mean([r['final_accuracy']    for r in grp_data]):>7.4f} "
                      f"{np.mean([r['total_comm_mb']     for r in grp_data]):>9.2f} "
                      f"{np.mean([r['total_comp_sec']    for r in grp_data]):>10.2f} "
                      f"{np.mean([r['re_communication']  for r in grp_data]):>10.4f} "
                      f"{np.mean([r['re_computation']    for r in grp_data]):>10.4f} "
                      f"{np.mean([r['re_parameter']      for r in grp_data]):>10.4f}")

        print(f"\n  By Head Type:")
        print(f"  {'Head':<8} {'N':>3} "
              f"{'Acc':>7} {'CommMB':>9} {'CompSec':>10} "
              f"{'RE_Comm':>10} {'RE_Comp':>10} {'RE_Param':>10}")
        print(f"  {'─'*70}")

        for head in ['DUAL', 'SINGLE']:
            head_data = [r for r in self.re_summary.values()
                         if r['head_type'] == head]
            if head_data:
                print(f"  {head:<8} {len(head_data):>3} "
                      f"{np.mean([r['final_accuracy']    for r in head_data]):>7.4f} "
                      f"{np.mean([r['total_comm_mb']     for r in head_data]):>9.2f} "
                      f"{np.mean([r['total_comp_sec']    for r in head_data]):>10.2f} "
                      f"{np.mean([r['re_communication']  for r in head_data]):>10.4f} "
                      f"{np.mean([r['re_computation']    for r in head_data]):>10.4f} "
                      f"{np.mean([r['re_parameter']      for r in head_data]):>10.4f}")

        print(f"\n  By Severity:")
        print(f"  {'Severity':<10} {'N':>3} "
              f"{'Acc':>7} {'RE_Comm':>10} "
              f"{'RE_Comp':>10} {'RE_Param':>10}")
        print(f"  {'─'*55}")

        for sev in ['LOW', 'MODERATE', 'HIGH', 'CRITICAL']:
            sev_data = [r for r in self.re_summary.values()
                        if r['severity'] == sev]
            if sev_data:
                print(f"  {sev:<10} {len(sev_data):>3} "
                      f"{np.mean([r['final_accuracy']   for r in sev_data]):>7.4f} "
                      f"{np.mean([r['re_communication'] for r in sev_data]):>10.4f} "
                      f"{np.mean([r['re_computation']   for r in sev_data]):>10.4f} "
                      f"{np.mean([r['re_parameter']     for r in sev_data]):>10.4f}")

    def print_client_model_sizes(self):
        if not self.client_model_sizes:
            print("  No client model sizes recorded.")
            return

        print(f"\n  CLIENT MODEL SIZE BREAKDOWN")
        print(f"  {'─'*75}")
        print(f"  {'User':<10} {'Sev':<9} {'Head':<6} {'Config':<14} "
              f"{'Total':>8} {'Backbone':>9} {'Head':>8} {'MB':>7}")
        print(f"  {'─'*75}")

        for uid in sorted(self.client_model_sizes.keys()):
            s = self.client_model_sizes[uid]
            print(f"  {uid:<10} {s['severity']:<9} {s['head_type']:<6} "
                  f"{s['head_config']:<14} "
                  f"{s['total_params']:>8,} "
                  f"{s['backbone_params']:>9,} "
                  f"{s['head_params']:>8,} "
                  f"{s['memory_mb']:>7.3f}")

        print(f"  {'─'*75}")

        all_sizes    = list(self.client_model_sizes.values())
        dual_sizes   = [s for s in all_sizes if s['head_type'] == 'DUAL']
        single_sizes = [s for s in all_sizes if s['head_type'] == 'SINGLE']

        print(f"\n  Summary:")
        if dual_sizes:
            dual_params = [s['total_params'] for s in dual_sizes]
            dual_head   = [s['head_params']  for s in dual_sizes]
            print(f"    DUAL   ({len(dual_sizes)} clients): "
                  f"total={np.mean(dual_params):,.0f}"
                  f"±{np.std(dual_params):,.0f} params | "
                  f"head={np.mean(dual_head):,.0f}"
                  f"±{np.std(dual_head):,.0f} params | "
                  f"avg={np.mean([s['memory_mb'] for s in dual_sizes]):.3f} MB")
        if single_sizes:
            single_params = [s['total_params'] for s in single_sizes]
            single_head   = [s['head_params']  for s in single_sizes]
            print(f"    SINGLE ({len(single_sizes)} clients): "
                  f"total={np.mean(single_params):,.0f}"
                  f"±{np.std(single_params):,.0f} params | "
                  f"head={np.mean(single_head):,.0f}"
                  f"±{np.std(single_head):,.0f} params | "
                  f"avg={np.mean([s['memory_mb'] for s in single_sizes]):.3f} MB")

        if dual_sizes and single_sizes:
            avg_dual   = np.mean([s['total_params'] for s in dual_sizes])
            avg_single = np.mean([s['total_params'] for s in single_sizes])
            overhead   = ((avg_dual - avg_single) / avg_single) * 100
            print(f"\n    Dual-head overhead vs single-head: "
                  f"{overhead:+.2f}% "
                  f"({avg_dual - avg_single:+,.0f} params)")

    def get_summary(self):
        if not self.round_metrics:
            return {'error': 'No data collected'}

        durations     = [m['duration']     for m in self.round_metrics.values()
                         if 'duration'     in m]
        memory_deltas = [m['memory_delta'] for m in self.round_metrics.values()
                         if 'memory_delta' in m]
        peak_memories = [m['peak_memory']  for m in self.round_metrics.values()
                         if 'peak_memory'  in m]

        return {
            'baseline_memory_mb'  : self.baseline_memory,
            'total_rounds'        : len(self.round_metrics),
            'total_time'          : np.sum(durations)      if durations     else 0,
            'avg_time_per_round'  : np.mean(durations)     if durations     else 0,
            'avg_memory_delta'    : np.mean(memory_deltas) if memory_deltas else 0,
            'max_peak_memory'     : np.max(peak_memories)  if peak_memories else 0,
            'total_models'        : len(self.model_cache),
            'unique_architectures': len(self._measured_archs),
            'client_model_sizes'  : self.client_model_sizes,
            're_summary'          : self.re_summary,
        }

    def print_summary(self):
        summary = self.get_summary()
        if 'error' in summary:
            print(f"  {summary['error']}")
            return

        print(f"  RESOURCE SUMMARY")
        print(f"  Total Time   : {summary['total_time']:.2f}s "
              f"({summary['total_time']/60:.2f} min)")
        print(f"  Avg/Round    : {summary['avg_time_per_round']:.2f}s")
        print(f"  Total Rounds : {summary['total_rounds']}")
        print(f"\n  Memory:")
        print(f"  Baseline     : {summary['baseline_memory_mb']:.2f} MB")
        print(f"  Peak         : {summary['max_peak_memory']:.2f} MB")
        print(f"  Avg Delta    : {summary['avg_memory_delta']:.2f} MB/round")
        print(f"\n  Models:")
        print(f"  Total        : {summary['total_models']}")
        print(f"  Unique Archs : {summary['unique_architectures']}")

        if summary['client_model_sizes']:
            self.print_client_model_sizes()
        if summary['re_summary']:
            self.print_re_metrics()


In [ ]:
# Evaluate models
def evaluate_user_model_comprehensive(model, val_data, val_labels,
                                       client_severity, batch_size,
                                       global_model=None, use_ewc=True):
    from sklearn.metrics import (precision_score, recall_score,
                                  f1_score, confusion_matrix)

    n_samples  = len(val_data)
    n_batches  = max(1, (n_samples + batch_size - 1) // batch_size)

    all_preds  = []
    all_labels = []

    # WITH EWC + HIGH/MOD/CRIT → use global_model if available
    # WITH EWC + LOW           → use local model personal_output
    # WITHOUT EWC              → use local model global_output
    use_global_model = (use_ewc and
                        client_severity in GLOBAL_SEVERITIES and
                        global_model is not None)

    eval_model = global_model if use_global_model else model

    try:
        for batch_idx in range(n_batches):
            start   = batch_idx * batch_size
            end     = min(start + batch_size, n_samples)

            batch_x = val_data[start:end]
            batch_y = val_labels[start:end]

            preds = eval_model(batch_x, training=False)

            # Pick correct output head
            if isinstance(preds, dict):
                if use_ewc and client_severity in PERSONAL_SEVERITIES:
                    # EWC ON + LOW → personal head
                    preds = preds.get('personal_output',
                                      preds.get('global_output'))
                else:
                    preds = preds['global_output']
            elif isinstance(preds, (list, tuple)):
                out_names = eval_model.output_names
                if (use_ewc and
                        client_severity in PERSONAL_SEVERITIES and
                        'personal_output' in out_names):
                    preds = preds[out_names.index('personal_output')]
                else:
                    idx = (out_names.index('global_output')
                           if 'global_output' in out_names else 0)
                    preds = preds[idx]

            all_preds.append(np.argmax(preds.numpy(), axis=1))
            all_labels.append(np.argmax(batch_y, axis=1))

            del batch_x, batch_y, preds

        all_preds  = np.concatenate(all_preds)
        all_labels = np.concatenate(all_labels)

        accuracy   = float(np.mean(all_preds == all_labels))
        precision  = float(precision_score(
            all_labels, all_preds,
            average='weighted', zero_division=0))
        recall     = float(recall_score(
            all_labels, all_preds,
            average='weighted', zero_division=0))
        f1         = float(f1_score(
            all_labels, all_preds,
            average='weighted', zero_division=0))
        try:
            cm = confusion_matrix(all_labels, all_preds)
        except Exception:
            cm = None

        del all_preds, all_labels
        gc.collect()

        return {
            'accuracy'           : accuracy,
            'precision_weighted' : precision,
            'recall_weighted'    : recall,
            'f1_weighted'        : f1,
            'confusion_matrix'   : cm,
        }

    except tf.errors.ResourceExhaustedError as e:
        print(f"   GPU OOM in comprehensive eval: {e}")
        gc.collect()
        return {
            'accuracy': 0.0, 'precision_weighted': 0.0,
            'recall_weighted': 0.0, 'f1_weighted': 0.0,
            'confusion_matrix': None,
        }

    except Exception as e:
        print(f"    Eval error: {e}")
        gc.collect()
        return {
            'accuracy': 0.0, 'precision_weighted': 0.0,
            'recall_weighted': 0.0, 'f1_weighted': 0.0,
            'confusion_matrix': None,
        }

In [ ]:
# Main Train Loop
def train_fedprox_personalized_with_new_clients(
    group1_per_user, group1_splits, group2_per_user, group2_splits,
    input_shape, n_class,
    case='case1',
    num_rounds_initial=6,
    num_rounds_combined=29,
    local_epochs=2,
    batch_size=32,
    patience=100,
    mu=0.01,                   
    lambda_ewc=0.4,
    ewc_sample_ratio=0.49,
    fisher_batch_size=32,
    fisher_refresh_round=20,
    lambda_min=0.2,
    lambda_max=0.8,
    trial_interval=5,
    max_trials_per_user=9,
    num_trials=3,
    ewc_cache=None,
    use_ewc=True,
    severity_aware=True,
    weighting_strategy='hybrid',
    seed=None):

    run_label    = "WITH EWC" if use_ewc else "WITHOUT EWC (Ablation)"
    sev_label    = "Severity-Aware" if severity_aware else "ALL PERSONALIZED"
    total_rounds = num_rounds_initial + num_rounds_combined

    def _effective_severity(uid):      
        if severity_aware:
            return client_severities[uid]
        else:
            return 'LOW'

    print(f"  FEDPROX + PERSONALIZATION + EWC NEW CLIENTS")
    print(f"  Case: {case.upper()} | Seed: {seed} | Mode: {run_label}")
    print(f"  Severity Gating: {sev_label}") 
    print(f"  Phase 1: {num_rounds_initial} rounds (Group 1 only)")
    print(f"  Phase 2: {num_rounds_combined} rounds (Group 1 + Group 2)")
    print(f"  Total:   {total_rounds} rounds")
    print(f"  Local epochs: {local_epochs} | Batch: {batch_size}")
    print(f"  FedProx: μ={mu} (fixed)")

    if use_ewc:
        print(f"  EWC: λ_range=[{lambda_min}, {lambda_max}], "
              f"refresh@round {fisher_refresh_round}")
    else:
        print(f"  EWC: DISABLED — G2 clients use simple CE only")

    print(f"\n  Case Configuration:")
    if case == 'case1':
        print(f"    G1 (Initial): Mostly LOW severity")
        print(f"    G2 (New):     HIGH/MOD/CRIT")
    elif case == 'case2':
        print(f"    G1 (Initial): HIGH/MOD/CRIT")
        print(f"    G2 (New):     Mostly LOW severity")
    elif case == 'case3':
        print(f"    G1 (Initial): Mixed severity")
        print(f"    G2 (New):     Mixed severity")

    print(f"  Training Strategy:")
    if severity_aware:                                          
        print(f"    G1 → FedProx (μ={mu}, fixed)")
        print(f"       LOW: dual-head | HIGH/MOD/CRIT: single-head")
    else:                                                      
        print(f"    ALL → dual-head (severity gating DISABLED)")
        print(f"    FedProx μ={mu}")
    if use_ewc:
        print(f"    G2 → EWC (simple CE + EWC penalty)")
    else:
        print(f"    G2 → Simple CE only (ablation)")

    if weighting_strategy == 'hybrid':
        print(f"  FedAvg: backbone=quality, output=equal (HYBRID default)")
    elif weighting_strategy == 'equal':
        print(f"  FedAvg: backbone=equal, output=equal (ABLATION)")
    else:
        print(f"  FedAvg: backbone=quality, output=quality (ABLATION)")

    print(f"{'═'*80}")

    monitor      = ResourceMonitor()
    global_model = compile_global_model(input_shape, n_class)

    # aggregation strategy setup
    assert weighting_strategy in ('hybrid', 'equal', 'quality'), \
        f"Unknown weighting_strategy: {weighting_strategy}"

    def _get_agg_weights(round_contributors, contributor_splits):
        """Return (backbone_weights, output_weights) based on strategy."""
        quality_w = compute_quality_weights(
            list(round_contributors.keys()),
            contributor_splits,
            client_severity_metrics)
        equal_w = {uid: 1.0 / len(round_contributors)
                   for uid in round_contributors}
        if weighting_strategy == 'hybrid':
            return quality_w, equal_w
        elif weighting_strategy == 'equal':
            return equal_w, equal_w
        else:  # 'quality'
            return quality_w, quality_w

    _fisher_agg_map = {
        'hybrid':  'quality_weighted',
        'equal':   'equal',
        'quality': 'quality_weighted',
    }
    fisher_aggregation = _fisher_agg_map[weighting_strategy]

    if use_ewc:
        print(f"  Fisher aggregation: {fisher_aggregation}")

    # Severity Detection 
    print(f"\n  Severity Detection:")
    client_severities       = {}
    client_severity_metrics = {}
    all_per_user            = {**group1_per_user, **group2_per_user}

    for uid in sorted(all_per_user.keys()):
        s              = all_per_user[uid]
        val_key_labels = ('val_labels' if 'val_labels' in s
                          else 'test_labels')
        severity, metrics = detect_severity_distribution_only(
            s['train_labels'], s[val_key_labels], n_class)
        client_severities[uid]       = severity
        client_severity_metrics[uid] = metrics
        group_label = "G1" if uid in group1_per_user else "G2"
        eff_sev   = _effective_severity(uid)
        head_type   = ("DUAL" if severity in PERSONAL_SEVERITIES
                        else "SINGLE")
        print(f"    {uid:8s} [{group_label}]: {severity:8s} "
              f"[{head_type:6s}] "
              f"(score={metrics['score']:2.0f}, "
              f"overlap={metrics['overlap_ratio']:.2f})")

    print(f"\n  Severity Distribution by Group:")
    for grp_label, grp_dict in [('G1', group1_per_user),
                                  ('G2', group2_per_user)]:
        sev_counts = {}
        for uid in grp_dict:
            sev = client_severities[uid]
            sev_counts[sev] = sev_counts.get(sev, 0) + 1
        print(f"    {grp_label}: {dict(sev_counts)}")
    if not severity_aware:      
        print(f"    ⚠️  severity_aware=False → ALL clients get dual-head")

    # G1 model creation 
    print(f"\n  Creating Group 1 Client Models:")
    group1_models = {}
    for uid in sorted(group1_per_user.keys()):
        group1_models[uid] = create_model_for_user(
            uid, input_shape, n_class,
            client_severities[uid],
            global_model,
            use_ewc=use_ewc,
            severity_aware=severity_aware)
        monitor.measure_client_model(
            uid, group1_models[uid], client_severities[uid])

    #  Tracking variables 
    per_round_per_user_val_metrics  = {}
    per_round_per_user_test_metrics = {}
    per_round_global_acc            = []

    user_best_val_acc = {uid: 0.0   for uid in all_per_user}
    user_patience_ctr = {uid: 0     for uid in all_per_user}
    user_stopped      = {uid: False for uid in all_per_user}
    trial_counts      = {uid: 0     for uid in all_per_user}

    fisher_numpy            = None
    layer_to_weights        = None
    reference_weights_numpy = None
    group2_models           = {}
    adaptive_lambdas        = {}
    zero_shot_metrics       = {}                    


    # PHASE 1: Initial Clients Only
    print(f"\n{'═'*70}")
    print(f"  PHASE 1: Initial Clients Only "
          f"(Rounds 1–{num_rounds_initial})")
    print(f"  Strategy: FedProx (μ={mu}) for all G1 clients")
    print(f"{'═'*70}")

    for round_num in range(num_rounds_initial):
        monitor.mark_round_start(round_num)
        print(f"\n{'─'*65}\n  Round {round_num+1}/{total_rounds} "
              f"[PHASE 1 - {case.upper()}]\n{'─'*65}")

        global_backbone  = get_backbone_weights(global_model)
        global_output_ws = get_global_output_weights(global_model)

        # Trials G1 
        if round_num > 0 and round_num % trial_interval == 0:
            for uid in sorted(group1_per_user.keys()):
                if should_run_trial(round_num, uid, client_severities,
                                    trial_counts, trial_interval,
                                    max_trials_per_user,
                                    use_ewc=use_ewc):
                    s              = group1_per_user[uid]
                    val_key_data   = ('val_data'   if 'val_data'   in s
                                      else 'test_data')
                    val_key_labels = ('val_labels' if 'val_labels' in s
                                      else 'test_labels')
                    trial_splits = {
                        'train_data':   s['train_data'],
                        'train_labels': s['train_labels'],
                        'val_data':     s[val_key_data],   
                        'val_labels':   s[val_key_labels], 
                    }
                    new_model, improved = run_lightweight_trial(
                        uid,
                        group1_models[uid],
                        trial_splits,
                        global_model,
                        input_shape,
                        n_class,
                        client_severities[uid],
                        trial_counts,         
                        num_trials=num_trials,
                        quick_epochs=2,
                        batch_size=batch_size)
                    if improved:
                        del group1_models[uid]
                        group1_models[uid] = new_model
                        monitor.measure_client_model(
                            uid, group1_models[uid],
                            client_severities[uid])
                    trial_counts[uid] += 1
                gc.collect()

        # Local training G1
        round_contributors = {}
        for uid in sorted(group1_per_user.keys()):
            if user_stopped[uid]:
                continue
            splits         = group1_per_user[uid]
            val_key_data   = ('val_data'   if 'val_data'   in splits
                              else 'test_data')
            val_key_labels = ('val_labels' if 'val_labels' in splits
                              else 'test_labels')

            try:
                set_backbone_weights(group1_models[uid], global_backbone)
                if global_output_ws:
                    set_global_output_weights(
                        group1_models[uid], global_output_ws)
                    
                sev = _effective_severity(uid) 
                train_start = time.time()
                history = train_local_fedprox_unified_v2(
                    group1_models[uid],
                    global_backbone,
                    splits['train_data'],
                    splits['train_labels'],
                    mu,                 
                    local_epochs,
                    batch_size,
                    splits[val_key_data],
                    splits[val_key_labels],
                    client_severity=sev,
                    use_ewc=use_ewc)
                train_end = time.time()

                monitor.log_client_computation(
                    round_num, uid, train_start, train_end)
                monitor.log_client_communication(
                    round_num, uid, group1_models[uid])

                val_acc   = (history['val_accuracy'][-1]
                             if history['val_accuracy'] else 0.0)
                train_acc = history['train_accuracy'][-1]
                round_contributors[uid] = group1_models[uid]

                head = "D" if sev in PERSONAL_SEVERITIES else "S"
                print(f"    {uid:8s}[{sev:4s}|{head}] "
                      f"tr={train_acc:.4f} val={val_acc:.4f} "
                      f"μ={mu:.5f}")

                if val_acc > user_best_val_acc[uid]:
                    user_best_val_acc[uid] = val_acc
                    user_patience_ctr[uid] = 0
                else:
                    user_patience_ctr[uid] += 1
                    if user_patience_ctr[uid] >= patience:
                        user_stopped[uid] = True
                        print(f"    {uid}: ⏹ early stop "
                              f"(best={user_best_val_acc[uid]:.4f})")

            except Exception as e:
                print(f"    {uid}:  {e}")
                import traceback; traceback.print_exc()

        gc.collect()   

        # FedAvg Phase 1 
        if round_contributors:
            bb_weights, out_weights = _get_agg_weights(
                round_contributors, group1_per_user)

            if round_num % 10 == 0:
                print(f"\n    FedAvg Weights "
                      f"(Phase 1 - {case.upper()} "
                      f"| strategy={weighting_strategy}):")
                for uid in sorted(round_contributors.keys()):
                    print(f"      {uid}: "
                          f"backbone={bb_weights[uid]:.4f}, "
                          f"global_out={out_weights[uid]:.4f} "
                          f"({client_severities[uid]})")

            averaged_backbone = federated_avg_backbone(
                round_contributors, bb_weights,
                verbose=(round_num % 10 == 0))
            if averaged_backbone:
                set_backbone_weights(global_model, averaged_backbone)

            avg_global_output = federated_avg_global_output(
                round_contributors, out_weights,
                verbose=(round_num % 10 == 0))
            if avg_global_output:
                set_global_output_weights(global_model, avg_global_output)

        # Evaluation Phase 1
        per_round_per_user_val_metrics[round_num]  = {}
        per_round_per_user_test_metrics[round_num] = {}

        for uid in group1_per_user:
            sev            = _effective_severity(uid) 
            splits         = group1_per_user[uid]
            val_key_data   = ('val_data'   if 'val_data'   in splits
                              else 'test_data')
            val_key_labels = ('val_labels' if 'val_labels' in splits
                              else 'test_labels')

            per_round_per_user_val_metrics[round_num][uid] = \
                evaluate_user_model_comprehensive(
                    group1_models[uid],
                    splits[val_key_data],
                    splits[val_key_labels],
                    sev, batch_size,
                    global_model=global_model,
                    use_ewc=use_ewc)

            per_round_per_user_test_metrics[round_num][uid] = \
                evaluate_user_model_comprehensive(
                    group1_models[uid],
                    splits['test_data'],
                    splits['test_labels'],
                    sev, batch_size,
                    global_model=global_model,
                    use_ewc=use_ewc)

            monitor.log_client_accuracy(
                round_num, uid,
                per_round_per_user_test_metrics[round_num][uid]['accuracy'])

        print(f"\n  Post-FedAvg Metrics [Phase 1 - {case.upper()}]:")
        print(f"  {'User':<10} {'Sev':<8} {'H':<2} "
              f"{'Acc':>6} {'Prec':>6} {'Rec':>6} {'F1':>6}")
        print(f"  {'─'*52}")
        for uid in sorted(group1_per_user.keys()):
            sev     = _effective_severity(uid) 
            head    = "D" if sev in PERSONAL_SEVERITIES else "S"
            metrics = per_round_per_user_test_metrics[round_num][uid]
            print(f"  {uid:<10} {sev:<8} {head:<2} "
                  f"{metrics['accuracy']:>6.4f} "
                  f"{metrics['precision_weighted']:>6.4f} "
                  f"{metrics['recall_weighted']:>6.4f} "
                  f"{metrics['f1_weighted']:>6.4f}")

        global_acc = evaluate_with_generator(
            global_model,
            group1_splits['test_data'],
            group1_splits['test_labels'],
            batch_size)
        per_round_global_acc.append(global_acc)

        all_test_acc = [
            m['accuracy']
            for m in per_round_per_user_test_metrics[round_num].values()]
        print(f"\n    G1 TEST: "
              f"{np.mean(all_test_acc):.4f}"
              f"±{np.std(all_test_acc):.4f} | "
              f"Global pool: {global_acc:.4f}")
        print(f"    Stopped: "
              f"{sum(user_stopped[u] for u in group1_per_user)}/"
              f"{len(group1_per_user)}")

        monitor.mark_round_end(round_num)
        gc.collect()


    # TRANSITION
    print(f"\n{'═'*70}")
    print(f"  TRANSITION: Preparing for Group 2 "
          f"({case.upper()}) [{run_label}]")
    print(f"{'═'*70}")

    shared_layer_indices = []
    for idx, layer in enumerate(global_model.layers):
        if any(bb in layer.name for bb in BACKBONE_LAYER_NAMES):
            shared_layer_indices.append(idx)

    if use_ewc:
        print(f"\n  Computing Federated Fisher from Group 1...")
        print(f"  Fisher aggregation: {fisher_aggregation}") 
        fisher_numpy, layer_to_weights = create_fisher_federated(
            model=global_model,
            per_user_splits=group1_per_user,
            shared_layer_indices=shared_layer_indices,
            batch_size=fisher_batch_size,
            sample_ratio=ewc_sample_ratio,
            aggregation=fisher_aggregation,      
            severity_metrics=client_severity_metrics)

        reference_weights_numpy = global_model.get_weights()
        print(f"  Federated Fisher computed & reference weights stored")

        if ewc_cache is not None:
            ewc_cache.build(
                reference_weights_numpy=reference_weights_numpy,
                fisher_numpy=fisher_numpy,
                layer_to_weights=layer_to_weights,
                lambda_ewc=lambda_ewc)
            print(f"  EWCCache built: "
                  f"{len(ewc_cache._cache)} entries")
    else:
        print(f"\n  EWC DISABLED: Skipping Fisher computation")
        fisher_numpy            = None
        layer_to_weights        = None
        reference_weights_numpy = None

    gc.collect()

    # G2 model creation 
    print(f"\n  Creating Group 2 Client Models [{run_label}]:")
    for uid in sorted(group2_per_user.keys()):
        group2_models[uid] = create_model_for_user(
            uid, input_shape, n_class,
            client_severities[uid],
            global_model,
            use_ewc=use_ewc,
            severity_aware=severity_aware)
        monitor.measure_client_model(
            uid, group2_models[uid], client_severities[uid])

    # ZERO-SHOT EVALUATION — BEFORE any G2 training
    print(f"\n{'═'*70}")
    print(f"  ZERO-SHOT EVALUATION (Before G2 Training)")
    print(f"  Global model trained on: G1 clients only "
          f"({num_rounds_initial} rounds)")
    print(f"  Evaluating on: G2 clients (never trained)")
    if case == 'case3':
        print(f"  Cross-room: G1=room1+room2 → G2=room3 (new environment)")
    print(f"{'═'*70}")

    
    print(f"\n  Global Model → G2 test data (pure zero-shot):")
    print(f"  {'User':<10} {'Sev':<9} {'Acc':>7} {'Prec':>7} "
          f"{'Rec':>7} {'F1':>7}")
    print(f"  {'─'*50}")

    for uid in sorted(group2_per_user.keys()):
        splits = group2_per_user[uid]
        sev    = client_severities[uid]

        zs_global = evaluate_user_model_comprehensive(
            global_model,
            splits['test_data'],
            splits['test_labels'],
            'GLOBAL',                    
            batch_size,
            global_model=None,
            use_ewc=False)

        zero_shot_metrics[uid] = {
            'global_model': zs_global,
            'severity':     sev,
        }

        print(f"  {uid:<10} {sev:<9} "
              f"{zs_global['accuracy']:>7.4f} "
              f"{zs_global['precision_weighted']:>7.4f} "
              f"{zs_global['recall_weighted']:>7.4f} "
              f"{zs_global['f1_weighted']:>7.4f}")


    # Global model → combined G2 test pool 
    zs_combined_acc = evaluate_with_generator(
        global_model,
        group2_splits['test_data'],
        group2_splits['test_labels'],
        batch_size)

    print(f"\n  D) Global Model → Combined G2 test pool: "
          f"{zs_combined_acc:.4f}")

    # G1 reference at this point 
    zs_g1_acc = evaluate_with_generator(
        global_model,
        group1_splits['test_data'],
        group1_splits['test_labels'],
        batch_size)

    # Zero-shot summary 
    zs_accs = [zero_shot_metrics[uid]['global_model']['accuracy']
               for uid in group2_per_user]
    zs_f1s  = [zero_shot_metrics[uid]['global_model']['f1_weighted']
               for uid in group2_per_user]

    print(f"\n  ZERO-SHOT SUMMARY:")
    print(f"  {'─'*55}")
    print(f"  Global → G2 (per-user avg): "
          f"acc={np.mean(zs_accs):.4f}±{np.std(zs_accs):.4f} | "
          f"F1={np.mean(zs_f1s):.4f}±{np.std(zs_f1s):.4f}")
    print(f"  Global → G2 (pooled):       "
          f"acc={zs_combined_acc:.4f}")
    print(f"  Global → G1 (pooled):       "
          f"acc={zs_g1_acc:.4f}")
    print(f"  Cross-domain gap (G1-G2):   "
          f"{zs_g1_acc - zs_combined_acc:+.4f}")

    if case == 'case3':
        print(f"\n  Cross-Room Generalization:")
        print(f"    Room1+Room2 → Room3 zero-shot: "
              f"{zs_combined_acc:.4f}")
        print(f"    This measures how well activities learned in "
              f"room1/room2")
        print(f"    transfer to the unseen room3 environment.")

    print(f"{'═'*70}\n")

    gc.collect()

    # Adaptive lambda 
    print(f"\n  Computing adaptive λ for Group 2 "
          f"[range: {lambda_min}–{lambda_max}]:")
    print(f"  {'─'*55}")

    for uid, splits in group2_per_user.items():
        if use_ewc:
            val_key_data   = ('val_data'   if 'val_data'   in splits
                              else 'test_data')
            val_key_labels = ('val_labels' if 'val_labels' in splits
                              else 'test_labels')
            sev            = _effective_severity(uid)
            current_acc = evaluate_user_model(
                group2_models[uid],
                splits[val_key_data],
                splits[val_key_labels],
                sev,
                batch_size,
                global_model=global_model,
                use_ewc=True)

            n_samples        = len(splits['train_data'])
            size_factor      = float(
                np.clip(1.0 - (n_samples / 2000.0), 0.0, 1.0))
            class_counts     = np.sum(splits['train_labels'], axis=0)
            active_classes   = int(np.sum(class_counts > 0))
            n_total_classes  = splits['train_labels'].shape[1]
            diversity_factor = 1.0 - (active_classes / n_total_classes)

            raw_lambda = (lambda_ewc
                          + 0.2 * (1.0 - current_acc)
                          + 0.2 * size_factor
                          + 0.1 * diversity_factor)
            adaptive_lambdas[uid] = float(
                np.clip(round(raw_lambda, 3), lambda_min, lambda_max))
            print(f"    {uid}: λ={adaptive_lambdas[uid]:.3f} "
                  f"[acc={current_acc:.3f} | n={n_samples} | "
                  f"classes={active_classes}/{n_total_classes}]")
        else:
            adaptive_lambdas[uid] = 0.0
            print(f"    {uid}: λ=0.000 [EWC disabled]")

    if use_ewc:
        lam_vals = list(adaptive_lambdas.values())
        print(f"  {'─'*55}")
        print(f"  λ range: {min(lam_vals):.3f} – {max(lam_vals):.3f} "
              f"(avg={np.mean(lam_vals):.3f})")


    # PHASE 2: All Clients Together
    print(f"\n{'═'*70}")
    print(f"  PHASE 2: All Clients ({case.upper()}) [{run_label}]")
    print(f"  Rounds {num_rounds_initial+1}–{total_rounds}")
    if use_ewc:
        print(f"  Fisher refresh scheduled at Round "
              f"{fisher_refresh_round}")
        print(f"  G1 → FedProx | G2 → Simple CE + EWC penalty")
    else:
        print(f"  G1 → FedProx | G2 → Simple CE only (no EWC)")
    print(f"{'═'*70}")

    combined_test_data   = np.concatenate(
        [group1_splits['test_data'],
         group2_splits['test_data']],   axis=0)
    combined_test_labels = np.concatenate(
        [group1_splits['test_labels'],
         group2_splits['test_labels']], axis=0)

    for round_num in range(num_rounds_initial, total_rounds):
        monitor.mark_round_start(round_num)
        current_round = round_num + 1
        print(f"\n{'─'*65}\n  Round {current_round}/{total_rounds} "
              f"[PHASE 2 - {case.upper()} | {run_label}]\n{'─'*65}")

        global_backbone  = get_backbone_weights(global_model)
        global_output_ws = get_global_output_weights(global_model)

        # Trials 
        if round_num > 0 and round_num % trial_interval == 0:
            # Trials G1
            for uid in sorted(group1_per_user.keys()):
                if should_run_trial(round_num, uid, client_severities,
                                    trial_counts, trial_interval,
                                    max_trials_per_user,
                                    use_ewc=use_ewc):
                    s              = group1_per_user[uid]
                    val_key_data   = ('val_data'   if 'val_data'   in s
                                      else 'test_data')
                    val_key_labels = ('val_labels' if 'val_labels' in s
                                      else 'test_labels')
                    trial_splits = {
                        'train_data':   s['train_data'],
                        'train_labels': s['train_labels'],
                        'val_data':     s[val_key_data],
                        'val_labels':   s[val_key_labels],
                    }
                    new_model, improved = run_lightweight_trial(
                        uid,
                        group1_models[uid],
                        trial_splits,
                        global_model,
                        input_shape,
                        n_class,
                        client_severities[uid],
                        trial_counts,
                        num_trials=num_trials,
                        quick_epochs=2,
                        batch_size=batch_size)
                    if improved:
                        del group1_models[uid]
                        group1_models[uid] = new_model
                        monitor.measure_client_model(
                            uid, group1_models[uid],
                            client_severities[uid])
                    trial_counts[uid] += 1
                gc.collect()

            # Trials G2
            for uid in sorted(group2_per_user.keys()):
                if should_run_trial(round_num, uid, client_severities,
                                    trial_counts, trial_interval,
                                    max_trials_per_user,
                                    use_ewc=use_ewc):
                    s              = group2_per_user[uid]
                    val_key_data   = ('val_data'   if 'val_data'   in s
                                      else 'test_data')
                    val_key_labels = ('val_labels' if 'val_labels' in s
                                      else 'test_labels')
                    trial_splits = {
                        'train_data':   s['train_data'],
                        'train_labels': s['train_labels'],
                        'val_data':     s[val_key_data],
                        'val_labels':   s[val_key_labels],
                    }
                    new_model, improved = run_lightweight_trial(
                        uid,
                        group2_models[uid],
                        trial_splits,
                        global_model,
                        input_shape,
                        n_class,
                        client_severities[uid],
                        trial_counts,
                        num_trials=num_trials,
                        quick_epochs=2,
                        batch_size=batch_size)
                    if improved:
                        del group2_models[uid]
                        group2_models[uid] = new_model
                        monitor.measure_client_model(
                            uid, group2_models[uid],
                            client_severities[uid])
                    trial_counts[uid] += 1
                gc.collect()

        # LOCAL TRAINING G1
        print(f"\n  [Group 1 — Initial Clients — FedProx μ={mu}]")
        round_contributors = {}

        for uid in sorted(group1_per_user.keys()):
            if user_stopped[uid]:
                continue
            splits         = group1_per_user[uid]
            val_key_data   = ('val_data'   if 'val_data'   in splits
                              else 'test_data')
            val_key_labels = ('val_labels' if 'val_labels' in splits
                              else 'test_labels')

            try:
                set_backbone_weights(group1_models[uid], global_backbone)
                if global_output_ws:
                    set_global_output_weights(
                        group1_models[uid], global_output_ws)
                    
                sev = _effective_severity(uid)
                train_start = time.time()
                history = train_local_fedprox_unified_v2(
                    group1_models[uid],
                    global_backbone,
                    splits['train_data'],
                    splits['train_labels'],
                    mu,
                    local_epochs,
                    batch_size,
                    splits[val_key_data],
                    splits[val_key_labels],
                    client_severity=sev,
                    use_ewc=use_ewc)
                train_end = time.time()

                monitor.log_client_computation(
                    round_num, uid, train_start, train_end)
                monitor.log_client_communication(
                    round_num, uid, group1_models[uid])

                val_acc   = (history['val_accuracy'][-1]
                             if history['val_accuracy'] else 0.0)
                train_acc = history['train_accuracy'][-1]
                round_contributors[uid] = group1_models[uid]

                sev  = client_severities[uid]
                head = "D" if sev in PERSONAL_SEVERITIES else "S"
                print(f"    {uid:8s}[{sev:4s}|{head}] "
                      f"tr={train_acc:.4f} val={val_acc:.4f} "
                      f"μ={mu:.5f}")

                if val_acc > user_best_val_acc[uid]:
                    user_best_val_acc[uid] = val_acc
                    user_patience_ctr[uid] = 0
                else:
                    user_patience_ctr[uid] += 1
                    if user_patience_ctr[uid] >= patience:
                        user_stopped[uid] = True
                        print(f"    {uid}: ⏹ early stop "
                              f"(best={user_best_val_acc[uid]:.4f})")

            except Exception as e:
                print(f"    {uid}: ❌ {e}")
                import traceback; traceback.print_exc()

        gc.collect()

        # LOCAL TRAINING G2
        print(f"\n  [Group 2 — New Clients — {run_label}]")

        for uid in sorted(group2_per_user.keys()):
            if user_stopped[uid]:
                continue
            splits         = group2_per_user[uid]
            val_key_data   = ('val_data'   if 'val_data'   in splits
                              else 'test_data')
            val_key_labels = ('val_labels' if 'val_labels' in splits
                              else 'test_labels')

            sev            = _effective_severity(uid)
            head = "D" if sev in PERSONAL_SEVERITIES else "S"

            try:
                set_backbone_weights(group2_models[uid], global_backbone)
                if global_output_ws:
                    set_global_output_weights(
                        group2_models[uid], global_output_ws)

                train_start = time.time()
                history = train_local_ewc_only(
                    local_model             = group2_models[uid],
                    global_backbone_weights = global_backbone,
                    train_data              = splits['train_data'],
                    train_labels            = splits['train_labels'],
                    epochs                  = local_epochs,
                    batch_size              = batch_size,
                    val_data                = splits[val_key_data],
                    val_labels              = splits[val_key_labels],
                    client_severity         = sev,
                    reference_weights_numpy = (reference_weights_numpy
                                               if use_ewc else None),
                    fisher_numpy            = (fisher_numpy
                                               if use_ewc else None),
                    layer_to_weights        = (layer_to_weights
                                               if use_ewc else None),
                    lambda_ewc              = (adaptive_lambdas[uid]
                                               if use_ewc else 0.0),
                    ewc_cache               = (ewc_cache
                                               if use_ewc else None),
                    use_ewc                 = use_ewc)
                train_end = time.time()

                monitor.log_client_computation(
                    round_num, uid, train_start, train_end)
                monitor.log_client_communication(
                    round_num, uid, group2_models[uid])

                val_acc   = (history['val_accuracy'][-1]
                             if history['val_accuracy'] else 0.0)
                train_acc = history['train_accuracy'][-1]
                ewc_loss  = history.get('ewc_loss',  [0.0])[-1]
                prox_loss = history.get('prox_loss', [0.0])[-1]
                round_contributors[uid] = group2_models[uid]

                if use_ewc:
                    if sev in GLOBAL_SEVERITIES:
                        print(f"    {uid:8s}[{sev:8s}|{head}] "
                              f"tr={train_acc:.4f} val={val_acc:.4f} "
                              f"ewc={ewc_loss:.6f} prox={prox_loss:.6f} "
                              f"[λ={adaptive_lambdas[uid]:.3f} μ={mu:.4f}]")
                    else:
                        print(f"    {uid:8s}[{sev:8s}|{head}] "
                              f"tr={train_acc:.4f} val={val_acc:.4f} "
                              f"ewc={ewc_loss:.6f} "
                              f"[λ={adaptive_lambdas[uid]:.3f}]")
                else:
                    print(f"    {uid:8s}[{sev:8s}|{head}] "
                          f"tr={train_acc:.4f} val={val_acc:.4f} "
                          f"[NO-EWC ablation]")

                if val_acc > user_best_val_acc[uid]:
                    user_best_val_acc[uid] = val_acc
                    user_patience_ctr[uid] = 0
                else:
                    user_patience_ctr[uid] += 1
                    if user_patience_ctr[uid] >= patience:
                        user_stopped[uid] = True
                        print(f"    {uid}: ⏹ early stop "
                              f"(best={user_best_val_acc[uid]:.4f})")

            except Exception as e:
                print(f"    {uid}: ❌ {e}")
                import traceback; traceback.print_exc()

        gc.collect()

        # FEDAVG PHASE 2
        if round_contributors:
            all_contributor_splits = {}
            for uid in round_contributors:
                if uid in group1_per_user:
                    all_contributor_splits[uid] = group1_per_user[uid]
                else:
                    all_contributor_splits[uid] = group2_per_user[uid]

            bb_weights, out_weights = _get_agg_weights(
                round_contributors, all_contributor_splits)

            if round_num % 10 == 0:
                print(f"\n    FedAvg Weights "
                      f"(Phase 2 - {case.upper()} "
                      f"| strategy={weighting_strategy}):")
                for uid in sorted(round_contributors.keys()):
                    grp = "G1" if uid in group1_per_user else "G2"
                    print(f"      {uid} [{grp}]: "
                          f"backbone={bb_weights[uid]:.4f}, "
                          f"global_out={out_weights[uid]:.4f} "
                          f"({client_severities[uid]})")

            averaged_backbone = federated_avg_backbone(
                round_contributors, bb_weights,
                verbose=(round_num % 10 == 0))
            if averaged_backbone:
                set_backbone_weights(global_model, averaged_backbone)

            avg_global_output = federated_avg_global_output(
                round_contributors, out_weights,
                verbose=(round_num % 10 == 0))
            if avg_global_output:
                set_global_output_weights(global_model, avg_global_output)

            if round_num % 10 == 0:
                print(f"    ✓ FedAvg complete: "
                      f"{len(round_contributors)} contributors "
                      f"(G1={sum(1 for u in round_contributors if u in group1_per_user)}, "
                      f"G2={sum(1 for u in round_contributors if u in group2_per_user)})")

        # Fisher Refresh 
        if use_ewc and current_round % fisher_refresh_round == 0:
            print(f"\n  {'─'*55}")
            print(f"  Federated Fisher Refresh at Round {current_round}")
            print(f"  {'─'*55}")

            fisher_numpy = refresh_fisher_federated(
                model=global_model,
                per_user_splits=group1_per_user,
                shared_layer_indices=shared_layer_indices,
                old_fisher_numpy=fisher_numpy,
                layer_to_weights=layer_to_weights,
                batch_size=fisher_batch_size,
                sample_ratio=ewc_sample_ratio,
                ema_alpha=0.3,
                aggregation='quality_weighted',
                severity_metrics=client_severity_metrics)

            reference_weights_numpy = global_model.get_weights()
            print(f"  Fisher refreshed & reference weights updated")

            if ewc_cache is not None:
                ewc_cache.refresh(
                    reference_weights_numpy=reference_weights_numpy,
                    fisher_numpy=fisher_numpy,
                    lambda_ewc=lambda_ewc)
                print(f"  EWCCache refreshed (base λ={lambda_ewc})")

            print(f"\n  Updating adaptive λ after Fisher refresh...")
            for uid, splits in group2_per_user.items():
                val_key_data   = ('val_data'   if 'val_data'   in splits
                                  else 'test_data')
                val_key_labels = ('val_labels' if 'val_labels' in splits
                                  else 'test_labels')
                sev            = _effective_severity(uid)
                current_acc = evaluate_user_model(
                    group2_models[uid],
                    splits[val_key_data],
                    splits[val_key_labels],
                    sev,
                    batch_size,
                    global_model=global_model,
                    use_ewc=True)

                n_samples        = len(splits['train_data'])
                size_factor      = float(
                    np.clip(1.0 - (n_samples / 2000.0), 0.0, 1.0))
                class_counts     = np.sum(splits['train_labels'], axis=0)
                active_classes   = int(np.sum(class_counts > 0))
                n_total_classes  = splits['train_labels'].shape[1]
                diversity_factor = 1.0 - (active_classes / n_total_classes)

                raw_lambda = (lambda_ewc
                              + 0.2 * (1.0 - current_acc)
                              + 0.2 * size_factor
                              + 0.1 * diversity_factor)
                adaptive_lambdas[uid] = float(
                    np.clip(round(raw_lambda, 3), lambda_min, lambda_max))
                print(f"    {uid}: λ={adaptive_lambdas[uid]:.3f} "
                      f"[acc={current_acc:.3f}]")

            lam_vals = list(adaptive_lambdas.values())
            print(f"  λ range after refresh: "
                  f"{min(lam_vals):.3f}–{max(lam_vals):.3f}")
            gc.collect()

        # EVALUATION PHASE 2
        per_round_per_user_val_metrics[round_num]  = {}
        per_round_per_user_test_metrics[round_num] = {}

        for uid in group1_per_user:
            sev            = _effective_severity(uid)
            splits         = group1_per_user[uid]
            val_key_data   = ('val_data'   if 'val_data'   in splits
                              else 'test_data')
            val_key_labels = ('val_labels' if 'val_labels' in splits
                              else 'test_labels')

            per_round_per_user_val_metrics[round_num][uid] = \
                evaluate_user_model_comprehensive(
                    group1_models[uid],
                    splits[val_key_data],
                    splits[val_key_labels],
                    sev, batch_size,
                    global_model=global_model,
                    use_ewc=use_ewc)
            per_round_per_user_test_metrics[round_num][uid] = \
                evaluate_user_model_comprehensive(
                    group1_models[uid],
                    splits['test_data'],
                    splits['test_labels'],
                    sev, batch_size,
                    global_model=global_model,
                    use_ewc=use_ewc)
            monitor.log_client_accuracy(
                round_num, uid,
                per_round_per_user_test_metrics[round_num][uid]['accuracy'])

        for uid in group2_per_user:
            sev            = _effective_severity(uid)
            splits         = group2_per_user[uid]
            val_key_data   = ('val_data'   if 'val_data'   in splits
                              else 'test_data')
            val_key_labels = ('val_labels' if 'val_labels' in splits
                              else 'test_labels')

            per_round_per_user_val_metrics[round_num][uid] = \
                evaluate_user_model_comprehensive(
                    group2_models[uid],
                    splits[val_key_data],
                    splits[val_key_labels],
                    sev, batch_size,
                    global_model=global_model,
                    use_ewc=use_ewc)
            per_round_per_user_test_metrics[round_num][uid] = \
                evaluate_user_model_comprehensive(
                    group2_models[uid],
                    splits['test_data'],
                    splits['test_labels'],
                    sev, batch_size,
                    global_model=global_model,
                    use_ewc=use_ewc)
            monitor.log_client_accuracy(
                round_num, uid,
                per_round_per_user_test_metrics[round_num][uid]['accuracy'])

        print(f"\n  Post-FedAvg Metrics "
              f"[Phase 2 - {case.upper()} | {run_label}]:")
        print(f"  {'User':<10} {'Grp':>4} {'Sev':>9} {'H':>2} "
              f"{'Acc':>6} {'Prec':>6} {'Rec':>6} {'F1':>6} {'λ':>7}")
        print(f"  {'─'*62}")
        for uid in sorted(all_per_user.keys()):
            sev_eff = _effective_severity(uid) 
            head    = "D" if sev_eff in PERSONAL_SEVERITIES else "S"
            grp     = "G1" if uid in group1_per_user else "G2"
            metrics = per_round_per_user_test_metrics[round_num][uid]
            lam_str = (f"{adaptive_lambdas.get(uid, 0):.3f}"
                       if grp == "G2" and use_ewc else "  —  ")
            print(f"  {uid:<10} {grp:>4} {sev:>9} {head:>2} "
                  f"{metrics['accuracy']:>6.4f} "
                  f"{metrics['precision_weighted']:>6.4f} "
                  f"{metrics['recall_weighted']:>6.4f} "
                  f"{metrics['f1_weighted']:>6.4f} {lam_str:>7}")

        global_acc = evaluate_with_generator(
            global_model,
            combined_test_data,
            combined_test_labels,
            batch_size)
        per_round_global_acc.append(global_acc)

        g1_test  = [per_round_per_user_test_metrics[round_num][u]['accuracy']
                    for u in group1_per_user]
        g2_test  = [per_round_per_user_test_metrics[round_num][u]['accuracy']
                    for u in group2_per_user]
        all_test = g1_test + g2_test

        low_test  = [
            per_round_per_user_test_metrics[round_num][u]['accuracy']
            for u in all_per_user
            if  _effective_severity(u) in PERSONAL_SEVERITIES] 
        high_test = [
            per_round_per_user_test_metrics[round_num][u]['accuracy']
            for u in all_per_user
            if  _effective_severity(u) in GLOBAL_SEVERITIES] 

        print(f"\n    G1 TEST:  "
              f"{np.mean(g1_test):.4f}±{np.std(g1_test):.4f}")
        print(f"    G2 TEST:  "
              f"{np.mean(g2_test):.4f}±{np.std(g2_test):.4f}")
        print(f"    ALL TEST: "
              f"{np.mean(all_test):.4f}±{np.std(all_test):.4f} | "
              f"Global: {global_acc:.4f}")
        if low_test:
            print(f"      LOW ({len(low_test)}): "
                  f"test={np.mean(low_test):.4f}"
                  f"±{np.std(low_test):.4f}")
        if high_test:
            print(f"      MOD/HIGH/CRIT ({len(high_test)}): "
                  f"test={np.mean(high_test):.4f}"
                  f"±{np.std(high_test):.4f}")
        print(f"    G1-G2 Gap: "
              f"{np.mean(g1_test) - np.mean(g2_test):+.4f}")
        print(f"    Stopped: "
              f"{sum(user_stopped.values())}/{len(all_per_user)}")

       
        if zero_shot_metrics:
            rounds_trained = round_num - num_rounds_initial + 1
            print(f"\n    Zero-Shot → Round {current_round} "
                  f"(after {rounds_trained} adaptation rounds):")
            for uid in sorted(group2_per_user.keys()):
                if uid in zero_shot_metrics:
                    zs_acc  = zero_shot_metrics[uid]['global_model']['accuracy']
                    cur_acc = per_round_per_user_test_metrics[round_num][uid]['accuracy']
                    delta   = cur_acc - zs_acc
                    print(f"      {uid}: ZS={zs_acc:.4f} → "
                          f"R{current_round}={cur_acc:.4f} "
                          f"(Δ={delta:+.4f})")

        monitor.mark_round_end(round_num)
        gc.collect()


    print(f"\n{'═'*70}")
    print(f"  TRAINING COMPLETE "
          f"({case.upper()} - Seed: {seed} - {run_label})")
    print(f"  Severity Gating: {sev_label}") 
    print(f"  Weighting Strategy: {weighting_strategy}") 
    print(f"{'═'*70}")
    print(f"  Phase 1: {num_rounds_initial} rounds (Group 1 only)")
    print(f"  Phase 2: {num_rounds_combined} rounds (All clients)")
    print(f"  Total:   {total_rounds} rounds")
    if use_ewc:
        print(f"  Fisher refresh: Round {fisher_refresh_round} "
              f"[FEDERATED + EMA]")

    last_round         = total_rounds - 1
    final_test_metrics = per_round_per_user_test_metrics[last_round]

    print(f"\n  Final Results by User:")
    print(f"  {'User':<10} {'Grp':>4} {'Sev':>9} {'Head':>6} "
          f"{'Acc':>6} {'Prec':>6} {'Rec':>6} {'F1':>6} "
          f"{'λ':>7} {'T':>3}")
    print(f"  {'─'*70}")
    for uid in sorted(all_per_user.keys()):
        sev_eff = _effective_severity(uid) 
        head    = "DUAL" if sev_eff in PERSONAL_SEVERITIES else "SINGLE"
        grp     = "G1" if uid in group1_per_user else "G2"
        t       = trial_counts.get(uid, 0)
        metrics = final_test_metrics[uid]
        lam_str = (f"{adaptive_lambdas.get(uid, 0):.3f}"
                   if grp == "G2" and use_ewc else "  —  ")
        print(f"  {uid:<10} {grp:>4} {client_severities[uid]:>9} {head:>6} "
              f"{metrics['accuracy']:>6.4f} "
              f"{metrics['precision_weighted']:>6.4f} "
              f"{metrics['recall_weighted']:>6.4f} "
              f"{metrics['f1_weighted']:>6.4f} "
              f"{lam_str:>7} {t:>3}")

    g1_metrics       = [final_test_metrics[u] for u in group1_per_user]
    g2_metrics       = [final_test_metrics[u] for u in group2_per_user]
    all_test_metrics = list(final_test_metrics.values())

    print(f"\n  By Group (TEST):")
    print(f"    {'Group':<4} {'N':>3} {'Acc':>7} "
          f"{'Precision':>10} {'Recall':>10} {'F1':>10}")
    print(f"    {'─'*50}")
    for grp_label, grp_m in [('G1', g1_metrics),
                               ('G2', g2_metrics)]:
        print(f"    {grp_label:<4} {len(grp_m):>3} "
              f"{np.mean([m['accuracy']           for m in grp_m]):>7.4f} "
              f"{np.mean([m['precision_weighted'] for m in grp_m]):>10.4f} "
              f"{np.mean([m['recall_weighted']    for m in grp_m]):>10.4f} "
              f"{np.mean([m['f1_weighted']        for m in grp_m]):>10.4f}")

    print(f"\n  By Severity (TEST):")
    print(f"    {'Severity':<10} {'N':>3} {'Acc':>7} "
          f"{'Precision':>10} {'Recall':>10} {'F1':>10}")
    print(f"    {'─'*55}")
    for sev_level in ['LOW', 'MODERATE', 'HIGH', 'CRITICAL']:
        users = [uid for uid, s in client_severities.items()
                 if s == sev_level]
        if users:
            ml = [final_test_metrics[uid] for uid in users]
            print(f"    {sev_level:<10} {len(users):>3} "
                  f"{np.mean([m['accuracy']           for m in ml]):>7.4f} "
                  f"{np.mean([m['precision_weighted'] for m in ml]):>10.4f} "
                  f"{np.mean([m['recall_weighted']    for m in ml]):>10.4f} "
                  f"{np.mean([m['f1_weighted']        for m in ml]):>10.4f}")

    # ZERO-SHOT Final adaptation summary
    if zero_shot_metrics:
        print(f"\n  Zero-Shot → Final Adaptation (G2 clients):")
        print(f"  {'User':<10} {'ZS_Acc':>7} {'Final':>7} "
              f"{'Δ_Acc':>7} {'ZS_F1':>7} {'Final_F1':>8} {'Δ_F1':>7}")
        print(f"  {'─'*60}")

        zs_improvements = []
        zs_f1_improvements = []
        for uid in sorted(group2_per_user.keys()):
            if uid in zero_shot_metrics:
                zs  = zero_shot_metrics[uid]['global_model']
                fin = final_test_metrics[uid]
                d_acc = fin['accuracy'] - zs['accuracy']
                d_f1  = fin['f1_weighted'] - zs['f1_weighted']
                zs_improvements.append(d_acc)
                zs_f1_improvements.append(d_f1)

                print(f"  {uid:<10} "
                      f"{zs['accuracy']:>7.4f} "
                      f"{fin['accuracy']:>7.4f} "
                      f"{d_acc:>+7.4f} "
                      f"{zs['f1_weighted']:>7.4f} "
                      f"{fin['f1_weighted']:>8.4f} "
                      f"{d_f1:>+7.4f}")

        print(f"  {'─'*60}")
        print(f"  Avg Acc  improvement: "
              f"{np.mean(zs_improvements):+.4f}"
              f"±{np.std(zs_improvements):.4f}")
        print(f"  Avg F1   improvement: "
              f"{np.mean(zs_f1_improvements):+.4f}"
              f"±{np.std(zs_f1_improvements):.4f}")
        print(f"  Adaptation rounds:   {num_rounds_combined}")
        print(f"  Zero-shot pooled:    {zs_combined_acc:.4f}")
        print(f"  Final G2 pooled:     "
              f"{np.mean([final_test_metrics[u]['accuracy'] for u in group2_per_user]):.4f}")

        if case == 'case3':
            print(f"\n  Cross-Room Generalization Summary:")
            print(f"    Zero-shot (room1+2 → room3): "
                  f"{zs_combined_acc:.4f}")
            final_g2_acc = np.mean([final_test_metrics[u]['accuracy']
                                    for u in group2_per_user])
            print(f"    After adaptation:             "
                  f"{final_g2_acc:.4f}")
            print(f"    Improvement:                  "
                  f"{final_g2_acc - zs_combined_acc:+.4f}")
            print(f"    G1 at zero-shot point:        "
                  f"{zs_g1_acc:.4f}")
            final_g1_acc = np.mean([final_test_metrics[u]['accuracy']
                                    for u in group1_per_user])
            print(f"    G1 after adaptation:          "
                  f"{final_g1_acc:.4f}")
            print(f"    G1 change (forgetting?):      "
                  f"{final_g1_acc - zs_g1_acc:+.4f}")

    print(f"\n  Case-Specific Insights ({case.upper()}):")
    if case == 'case1':
        print(f"    - G1 (LOW) established simple baseline")
        print(f"    - G2 (HIGH/MOD) adapted via "
              f"{'EWC' if use_ewc else 'simple CE (no EWC)'}")
    elif case == 'case2':
        print(f"    - G1 (HIGH/MOD) established complex baseline")
        print(f"    - G2 (LOW severity) simple users joined via "
              f"{'EWC' if use_ewc else 'simple CE (no EWC)'}")
    elif case == 'case3':
        print(f"    - G1 (Mixed) heterogeneous initial group")
        print(f"    - G2 (Mixed) heterogeneous new group via "
              f"{'EWC' if use_ewc else 'simple CE (no EWC)'}")
    if not severity_aware:                       
        print(f"\n  ABLATION NOTE:")
        print(f"    severity_aware=False → ALL clients received dual-head")
        print(f"    No severity-based routing was applied")
        print(f"    μ={mu} (FedProx {'active' if mu > 0 else 'disabled'})")
    if weighting_strategy != 'hybrid':
        print(f"\n  AGGREGATION ABLATION NOTE:")
        print(f"    weighting_strategy='{weighting_strategy}'")
        if weighting_strategy == 'equal':
            print(f"    Both backbone AND global_output use equal weights")
            print(f"    Fisher aggregation: equal")
        else:
            print(f"    Both backbone AND global_output use quality weights")
            print(f"    Fisher aggregation: quality_weighted")

    print(f"\n  Overall TEST:")
    print(f"    Accuracy:  "
          f"{np.mean([m['accuracy'] for m in all_test_metrics]):.4f}"
          f"±{np.std([m['accuracy'] for m in all_test_metrics]):.4f}")
    print(f"    Precision: "
          f"{np.mean([m['precision_weighted'] for m in all_test_metrics]):.4f}"
          f"±{np.std([m['precision_weighted'] for m in all_test_metrics]):.4f}")
    print(f"    Recall:    "
          f"{np.mean([m['recall_weighted'] for m in all_test_metrics]):.4f}"
          f"±{np.std([m['recall_weighted'] for m in all_test_metrics]):.4f}")
    print(f"    F1-Score:  "
          f"{np.mean([m['f1_weighted'] for m in all_test_metrics]):.4f}"
          f"±{np.std([m['f1_weighted'] for m in all_test_metrics]):.4f}")
    print(f"  Global pool:   {per_round_global_acc[-1]:.4f}")
    print(f"  Total trials:  {sum(trial_counts.values())}")
    print(f"  Early stopped: "
          f"{sum(user_stopped.values())}/{len(all_per_user)}")

    g1_final_acc = [final_test_metrics[u]['accuracy']
                    for u in group1_per_user]
    g2_final_acc = [final_test_metrics[u]['accuracy']
                    for u in group2_per_user]

    if np.mean(g2_final_acc) >= np.mean(g1_final_acc) - 0.05:
        print(f"\n   New clients successfully adapted "
              f"(within 5% of initial clients)")
    else:
        print(f"\n    New clients still catching up — "
              f"consider more Phase 2 rounds")

    print(f"\n{'─'*70}")
    print(f"  Computing Resource Efficiency Metrics...")
    monitor.compute_re_metrics(
        client_severities = client_severities,
        group1_user_ids   = list(group1_per_user.keys()),
        group2_user_ids   = list(group2_per_user.keys()))
    monitor.print_summary()
    print(f"{'═'*80}\n")

    # ◄◄◄ ZERO-SHOT: Build return dict entries
    zs_return = {}
    if zero_shot_metrics:
        zs_return = {
            'zero_shot_metrics'       : zero_shot_metrics,
            'zero_shot_combined_acc'  : zs_combined_acc,
            'zero_shot_g1_acc'        : zs_g1_acc,
            'zero_shot_per_user_acc'  : {
                uid: zero_shot_metrics[uid]['global_model']['accuracy']
                for uid in zero_shot_metrics},
            'zero_shot_per_user_f1'   : {
                uid: zero_shot_metrics[uid]['global_model']['f1_weighted']
                for uid in zero_shot_metrics},
            'adaptation_improvement'  : {
                uid: (final_test_metrics[uid]['accuracy'] -
                      zero_shot_metrics[uid]['global_model']['accuracy'])
                for uid in zero_shot_metrics
                if uid in final_test_metrics},
        }

    return {
        'global_model'                   : global_model,
        'group1_models'                  : group1_models,
        'group2_models'                  : group2_models,
        'client_severities'              : client_severities,
        'client_severity_metrics'        : client_severity_metrics,
        'group1_user_ids'                : list(group1_per_user.keys()),
        'group2_user_ids'                : list(group2_per_user.keys()),
        'per_round_per_user_val_metrics' : per_round_per_user_val_metrics,
        'per_round_per_user_test_metrics': per_round_per_user_test_metrics,
        'per_round_per_user_val_acc'     : {
            r: {u: m['accuracy'] for u, m in metrics.items()}
            for r, metrics in per_round_per_user_val_metrics.items()},
        'per_round_per_user_test_acc'    : {
            r: {u: m['accuracy'] for u, m in metrics.items()}
            for r, metrics in per_round_per_user_test_metrics.items()},
        'per_round_global_acc'           : per_round_global_acc,
        'trial_counts'                   : trial_counts,
        'adaptive_lambdas'               : adaptive_lambdas,
        'fisher_numpy'                   : fisher_numpy,
        'layer_to_weights'               : layer_to_weights,
        'final_global_acc'               : per_round_global_acc[-1],
        'use_ewc'                        : use_ewc,
        'severity_aware'                 : severity_aware, 
        'weighting_strategy'             : weighting_strategy,
        'final_metrics_summary'          : {
            'all': {
                'accuracy' : np.mean([m['accuracy']
                                      for m in all_test_metrics]),
                'precision': np.mean([m['precision_weighted']
                                      for m in all_test_metrics]),
                'recall'   : np.mean([m['recall_weighted']
                                      for m in all_test_metrics]),
                'f1'       : np.mean([m['f1_weighted']
                                      for m in all_test_metrics]),
            },
            'g1': {
                'accuracy' : np.mean([m['accuracy']
                                      for m in g1_metrics]),
                'precision': np.mean([m['precision_weighted']
                                      for m in g1_metrics]),
                'recall'   : np.mean([m['recall_weighted']
                                      for m in g1_metrics]),
                'f1'       : np.mean([m['f1_weighted']
                                      for m in g1_metrics]),
            },
            'g2': {
                'accuracy' : np.mean([m['accuracy']
                                      for m in g2_metrics]),
                'precision': np.mean([m['precision_weighted']
                                      for m in g2_metrics]),
                'recall'   : np.mean([m['recall_weighted']
                                      for m in g2_metrics]),
                'f1'       : np.mean([m['f1_weighted']
                                      for m in g2_metrics]),
            },
        },
        're_summary'        : monitor.re_summary,
        'client_model_sizes': monitor.client_model_sizes,
        'case'              : case,
        **zs_return,                        
    }

In [ ]:
# Run and save results
# JSON Encoder 
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):  return int(obj)
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.ndarray):  return obj.tolist()
        if isinstance(obj, np.bool_):    return bool(obj)
        return super().default(obj)

# Saves each single seed result 
def save_single_seed_result(result, seed, case, use_ewc,
                             save_dir='results'):

    os.makedirs(save_dir, exist_ok=True)

    ewc_tag  = 'ewc'  if use_ewc else 'noewc'
    filename = f"seed_{seed}_{case}_{ewc_tag}.json"
    filepath = os.path.join(save_dir, filename)
    skip_keys = {
        'global_model', 'group1_models', 'group2_models',
        'mu_controller_g1', 'mu_controller_g2',
        'fisher_numpy', 'layer_to_weights',
    }

    serializable = {}
    for key, value in result.items():
        if key in skip_keys:
            continue
        serializable[key] = value

    payload = {
        'metadata': {
            'seed'      : seed,
            'case'      : case,
            'use_ewc'   : use_ewc,
            'saved_at'  : datetime.now().isoformat(),
        },
        'result': serializable,
    }

    with open(filepath, 'w') as f:
        json.dump(payload, f, cls=NumpyEncoder, indent=2)

    size_mb = os.path.getsize(filepath) / (1024 * 1024)
    print(f"  Seed {seed} saved → {filepath}  ({size_mb:.2f} MB)")
    return filepath

# Train for one seed
def run_single_seed(seed, case, loader, input_shape, n_class,
                    use_ewc=True, save_dir='results', **train_kwargs):

    run_label = "WITH EWC" if use_ewc else "WITHOUT EWC"

    print(f"\n{'═'*80}")
    print(f"  RUNNING SEED {seed} | {case.upper()} | {run_label}")
    print(f"{'═'*80}")

    np.random.seed(seed)
    tf.random.set_seed(seed)
    random.seed(seed)
    seed_data = loader.load(seed)
    ewc_cache = EWCCache()

    filepath = None

    try:
        result = train_fedprox_personalized_with_new_clients(
            group1_per_user      = seed_data['group1_per_user'],
            group1_splits        = seed_data['group1_splits'],
            group2_per_user      = seed_data['group2_per_user'],
            group2_splits        = seed_data['group2_splits'],
            input_shape          = input_shape,
            n_class              = n_class,
            case                 = case,
            ewc_cache            = ewc_cache,
            use_ewc              = use_ewc,
            seed                 = seed,
            **train_kwargs)

        filepath = save_single_seed_result(
            result   = result,
            seed     = seed,
            case     = case,
            use_ewc  = use_ewc,
            save_dir = save_dir)

        print(f"  Seed {seed} complete and saved.")

    except Exception as e:
        print(f"   Seed {seed} FAILED: {e}")
        import traceback; traceback.print_exc()
        error_payload = {
            'metadata': {
                'seed'    : seed,
                'case'    : case,
                'use_ewc' : use_ewc,
                'saved_at': datetime.now().isoformat(),
            },
            'result': {'error': str(e)},
        }
        ewc_tag   = 'ewc' if use_ewc else 'noewc'
        err_path  = os.path.join(
            save_dir, f"seed_{seed}_{case}_{ewc_tag}_FAILED.json")
        os.makedirs(save_dir, exist_ok=True)
        with open(err_path, 'w') as f:
            json.dump(error_payload, f, indent=2)
        print(f"  Error record saved → {err_path}")

    finally:
        if 'result' in dir() and result is not None:
            for key in ['global_model', 'group1_models', 'group2_models',
                        'mu_controller_g1', 'mu_controller_g2',
                        'fisher_numpy']:
                if key in result:
                    obj = result.pop(key, None)
                    if obj is not None:
                        if key in ('group1_models', 'group2_models'):
                            for m in obj.values():
                                safe_model_delete(m)
                        else:
                            del obj

        ewc_cache._cache.clear()
        del seed_data
        gc.collect()
        K.clear_session()
        time.sleep(2)
        print(f" Memory cleaned after seed {seed}")

    return filepath


# Pipeline setup
CASE1        = 'case1'
USE_EWC     = True # False
INPUT_SHAPE = (38, 20, 20, 1)
N_CLASS     = 12
SAVE_DIR    = f'results/{CASE1}_{"ewc" if USE_EWC else "noewc"}'

TRAIN_KWARGS = dict(
    num_rounds_initial   = 10,
    num_rounds_combined  = 40,
    local_epochs         = 2,
    batch_size           = 32,
    patience             = 100,
    mu                   = 0.01,
    lambda_ewc           = 0.4,
    lambda_min           = 0.2,
    lambda_max           = 0.8,
    fisher_refresh_round = 15,
    trial_interval       = 5,
    max_trials_per_user  = 9,
    severity_aware       = True, # False
    weighting_strategy='hybrid' # Equal/Qualtiy
)

loader_case1 = LazySeedDataLoader(fl_clients, case=CASE1)

# Cell: Seed 42 
case1_42 = run_single_seed(
    seed        = 42,
    case        = CASE1,
    loader      = loader_case1,
    input_shape = INPUT_SHAPE,
    n_class     = N_CLASS,
    use_ewc     = USE_EWC,
    save_dir    = SAVE_DIR,
    **TRAIN_KWARGS)